# Занятие 4. Метрики, валидация и разведочный анализ данных

Три занятия подряд данные были чистыми: вино приезжает из sklearn без единого
пропуска, все признаки числовые. В реальной работе так не бывает. До модели
данные надо понять: что лежит в каждом столбце, где пропуски и почему,
какие значения невозможны, какие признаки связаны с ответом, а какие
подглядывают в него. Это и называется разведочным анализом, EDA.

Сегодня два датасета.

**Отток клиентов телеком-оператора.** Для каждого из семи тысяч клиентов
известны услуги, тип договора, способ оплаты, платежи и то, ушел ли он.
Задача компании — заранее найти тех, кто собирается уйти, и предложить им
что-нибудь. Это классическая бизнес-задача, и данные в ней немного грязные,
как и положено.

**Тексты из новостных групп.** Пять тематик, почти пять тысяч сообщений.
На них разберем, как превратить текст в числа: мешок слов, n-граммы, TF-IDF.

По ходу дела: как кодировать категориальные признаки, как устроены метрики
классификации и какие из них смотреть при дисбалансе классов, как устроены
разные схемы кросс-валидации, где в них прячутся утечки и как сохранить
обученную модель на диск. В конце — библиотека, которая строит большую часть
такого анализа автоматически.

**Как работать с ноутбуком.** Сегодня мы разбираемся в устройстве, поэтому
весь код уже написан. Запускайте ячейки сверху вниз и смотрите на выводы
и графики. В нескольких местах предложено поменять параметр и посмотреть,
что изменится. При первом запуске
нужен интернет: датасеты и списки стоп-слов `nltk` скачиваются и дальше лежат в кэше.
Если в системе настроен прокси, `nltk` через него не качает; тогда первый
запуск лучше сделать без прокси.

## 0. Подготовка

Кроме библиотек здесь две вспомогательные функции. `show_table` раскрашивает
числовые столбцы таблицы: чем больше число, тем ярче клетка, — так разницу
между моделями видно сразу, без чтения чисел. `plot_metric_bars` рисует
метрики нескольких моделей столбиками рядом.

In [ ]:
import io
import os
import warnings

import joblib
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib.patches import Patch, Rectangle
from matplotlib.ticker import MaxNLocator
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from nltk.tokenize import wordpunct_tokenize

from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_20newsgroups, fetch_openml
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix, f1_score,
    precision_recall_curve, precision_score, recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV, GroupKFold, KFold, LeaveOneOut, RepeatedStratifiedKFold, ShuffleSplit,
    StratifiedKFold, TimeSeriesSplit, cross_val_score, cross_validate, train_test_split,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, TargetEncoder

warnings.filterwarnings("ignore", category=ConvergenceWarning)

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

BLUE, BLACK, OCHRE, GREY = "#0072CE", "#0F1418", "#C98A3C", "#9CA3AF"
CLASS_COLORS = ["#E41A1C", "#4DAF4A", "#377EB8"]
MODEL_COLORS = [GREY, BLUE, OCHRE, "#4DAF4A", "#984EA3"]
SEED = 42


def show_table(frame, digits=3, axis=0):
    """Таблица с заливкой: большие значения ярче.

    axis=0 — каждый столбец со своей шкалой, axis=None — одна шкала на всю таблицу.
    """
    numeric = frame.select_dtypes("number").columns
    styled = frame.style.background_gradient(cmap="Greens", subset=numeric, axis=axis).format(precision=digits, subset=numeric)
    return styled


def plot_metric_bars(table, title):
    """Метрики нескольких моделей: группа столбиков на метрику, цвет на модель."""
    metrics = list(table.columns)
    width = 0.8 / len(table)
    fig, ax = plt.subplots(figsize=(max(8, 1.6 * len(metrics)), 4))
    for k, (name, row) in enumerate(table.iterrows()):
        positions = np.arange(len(metrics)) + (k - (len(table) - 1) / 2) * width
        bars = ax.bar(positions, row.to_numpy(dtype=float), width=width, color=MODEL_COLORS[k % len(MODEL_COLORS)], label=name)
        ax.bar_label(bars, fmt="%.2f", fontsize=7, padding=1)
    ax.set_xticks(np.arange(len(metrics)), metrics)
    ax.set_ylim(0, 1.05)
    ax.set_title(title)
    ax.legend(fontsize=9, loc="upper left", bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.show()

## 1. Данные: отток клиентов

Датасет лежит в открытом репозитории OpenML, sklearn умеет его скачивать.
Берем его как есть, без всякой предобработки: `raw` — это то, что пришло.

In [ ]:
raw = fetch_openml(data_id=42178, as_frame=True, parser="auto").frame
print("строк:", raw.shape[0], " столбцов:", raw.shape[1])
raw.sample(5, random_state=SEED)

Смотрим на случайные строки, а не на первые: первые строки файла часто
отсортированы, склеены из одного источника или вообще служебные, и по ним
легко сделать неверный вывод обо всем датасете.

Столбцы:

| столбец | что это |
|---|---|
| `gender`, `SeniorCitizen`, `Partner`, `Dependents` | пол, пенсионер ли, есть ли партнер и иждивенцы |
| `tenure` | сколько месяцев клиент с оператором |
| `PhoneService`, `MultipleLines` | телефония и несколько линий |
| `InternetService` и шесть столбцов после него | интернет и дополнительные услуги: защита, бэкап, поддержка, ТВ, кино |
| `Contract` | помесячный договор, на год или на два |
| `PaperlessBilling`, `PaymentMethod` | электронные счета и способ оплаты |
| `MonthlyCharges`, `TotalCharges` | платеж в месяц и сумма за все время |
| `Churn` | ушел ли клиент — целевая переменная |

## 2. Первый взгляд

Типы столбцов и память. Для числовых столбцов тип должен быть числовым,
для остальных — строкой или категорией. Любое расхождение — первый сигнал
о грязи.

In [ ]:
print(raw.dtypes.to_string())
print()
print(f"память: {raw.memory_usage(deep=True).sum() / 1e6:.1f} МБ")

`TotalCharges` — сумма денег, а хранится строкой. Значит, в столбце есть
значения, которые не превращаются в число. `SeniorCitizen` — это «да» или
«нет», но записан числом 0 или 1. С точки зрения смысла это категория,
с точки зрения pandas — число.

### Что показывает `describe`

Стандартная сводка pandas — `describe`. По умолчанию она берет только
числовые столбцы и для каждого считает число непустых значений, среднее,
стандартное отклонение, минимум, квартили и максимум.

In [ ]:
raw.describe()

Столбцов всего три: `SeniorCitizen`, `tenure` и `MonthlyCharges`.
`TotalCharges` сюда не попал, потому что хранится строкой. `count` везде
равен числу строк: в числовых столбцах пропусков нет.

С `include="all"` сводка берет и остальные столбцы. Для строк она считает
другое: сколько разных значений (`unique`), самое частое (`top`) и сколько
раз оно встретилось (`freq`).

In [ ]:
raw.describe(include="all").T

Для строковых столбцов `count` тоже равен числу строк: пропусков pandas
не видит. У `TotalCharges` больше шести тысяч разных значений, то есть
это число, записанное текстом, а самое частое значение — `' '`,
одиннадцать раз. К этому пробелу мы еще вернемся. Смешанную картину по всем столбцам сразу
удобнее держать в своей сводке, одна строка на столбец:

- `dtype` — тип столбца строкой;
- `n_missing` и `missing_share` — сколько пропусков `NaN` и какая это доля;
- `n_unique` — сколько разных значений без учета пропусков;
- `top` и `top_share` — самое частое значение и его доля среди всех строк.

In [ ]:
def column_report(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for column in df.columns:
        values = df[column]
        counts = values.value_counts(dropna=True)
        rows.append({
            "column": column,
            "dtype": str(values.dtype),
            "n_missing": int(values.isna().sum()),
            "missing_share": float(values.isna().mean()),
            "n_unique": int(values.nunique(dropna=True)),
            "top": counts.index[0] if len(counts) else None,
            "top_share": float(counts.iloc[0] / len(values)) if len(counts) else 0.0,
        })
    report = pd.DataFrame(rows).set_index("column")
    return report

In [ ]:
show_table(column_report(raw))

Пропусков в формальном смысле нет ни одного, и это как раз подозрительно.
Посмотрим на значения категорий и на то, что мешает `TotalCharges` стать
числом.

### Что лежит в категориях

Начнем с типа договора. Сначала просто несколько случайных значений столбца.

In [ ]:
raw["Contract"].sample(8)

Сколько раз встречается каждое значение, считает `value_counts`.
Результат — `Series`: в индексе значения, в самом столбце — сколько раз
каждое встретилось, по убыванию.

In [ ]:
contract_counts = raw["Contract"].value_counts()
contract_counts

В таком виде легко не заметить, как значения записаны на самом деле.
Словарь показывает строки целиком, с кавычками.

In [ ]:
contract_counts.to_dict()

Кавычки есть у одних значений и нет у других. То же для способа оплаты,
теперь сразу одной строкой.

In [ ]:
raw["PaymentMethod"].value_counts().to_dict()

### Что мешает `TotalCharges` стать числом

Первые значения выглядят как обычные числа, только записанные строками.

In [ ]:
raw["TotalCharges"].head()

`pd.to_numeric` переводит строки в числа. С `errors="coerce"` все,
что перевести нельзя, становится `NaN`, а не ошибкой.

In [ ]:
total_as_number = pd.to_numeric(raw["TotalCharges"], errors="coerce")
total_as_number.head()

In [ ]:
total_as_number.isna().sum()

Одиннадцать значений не превратились в число. Посмотрим на них.

In [ ]:
not_numbers = raw.loc[total_as_number.isna(), "TotalCharges"]
not_numbers

Выглядят как пустые, но пустыми не являются. `repr` показывает строку
такой, какая она есть, со всеми пробелами и кавычками.

In [ ]:
[repr(value) for value in not_numbers.unique()]

Две находки.

**Кавычки внутри значений.** Одни значения записаны как `'One year'`, другие
как `Month-to-month`. Это след формата ARFF, в котором хранится OpenML: строки
с пробелами там берутся в кавычки, и при чтении кавычки остались частью
значения. Для модели `'One year'` и `One year` были бы разными категориями.

**Пробел вместо числа.** У одиннадцати клиентов `TotalCharges` — это пробел
в тех же кавычках. Для pandas это не пропуск, а обычная строка, поэтому
сводки показали ноль пропусков. Настоящий пропуск должен быть `NaN`.

## 3. Чистка

Функция возвращает очищенную копию таблицы:

- во всех строковых столбцах убраны одинарные кавычки по краям значения;
- `TotalCharges` превращен в число, пробелы стали `NaN`;
- `SeniorCitizen` переведен в строки `"No"` и `"Yes"`, как остальные
  признаки «да / нет»;
- исходная таблица не меняется, строки не удаляются.

In [ ]:
def clean_telco(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()
    for column in cleaned.columns:
        if pd.api.types.is_string_dtype(cleaned[column]):
            cleaned[column] = cleaned[column].str.strip("'")
    cleaned["TotalCharges"] = pd.to_numeric(cleaned["TotalCharges"].str.strip(), errors="coerce")
    cleaned["SeniorCitizen"] = cleaned["SeniorCitizen"].map({0: "No", 1: "Yes"})
    return cleaned

In [ ]:
telco = clean_telco(raw)
print("форма до и после:", raw.shape, telco.shape)

Что было и что стало в каждом из трех мест.

In [ ]:
pd.DataFrame({"было": raw["Contract"].unique(), "стало": telco["Contract"].unique()})

In [ ]:
print("тип TotalCharges:", raw["TotalCharges"].dtype, "->", telco["TotalCharges"].dtype)
print("пропусков NaN:   ", raw["TotalCharges"].isna().sum(), "->", telco["TotalCharges"].isna().sum())

In [ ]:
pd.crosstab(raw["SeniorCitizen"].rename("было"), telco["SeniorCitizen"].rename("стало"))

### Почему пропуски именно здесь

Пропуск — тоже информация. Посмотрим, у кого `TotalCharges` пустой.

In [ ]:
telco[telco["TotalCharges"].isna()][["tenure", "MonthlyCharges", "TotalCharges", "Contract", "Churn"]]

У всех одиннадцати `tenure = 0`: это клиенты, которые только что подключились,
и счета им еще не выставляли. Пропуск не полностью случайный: он целиком
определяется другим, наблюдаемым признаком. Принято различать три механизма
пропусков:

- **полностью случайные** (MCAR, missing completely at random) — пропуск
  ни от чего не зависит, например, потерялась часть анкет;
- **случайные** (MAR, missing at random) — пропуск зависит только
  от наблюдаемых признаков, как здесь: он бывает только при `tenure = 0`;
- **неслучайные** (MNAR, missing not at random) — пропуск зависит от самого
  пропущенного значения, например, люди с большим доходом чаще не указывают
  доход. Самый неприятный случай: по остальным данным его не распознать.

От механизма зависит, чем заполнять. Здесь ответ подсказывает смысл: клиент
еще ничего не заплатил, значит сумма ноль. Проверим, что `TotalCharges`
вообще связан с другими столбцами так, как мы думаем: сумма за все время
должна быть примерно равна месячному платежу, умноженному на число месяцев.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
estimate = telco["MonthlyCharges"] * telco["tenure"]
axes[0].scatter(estimate, telco["TotalCharges"], s=6, alpha=0.3, color=BLUE)
axes[0].plot([0, 9000], [0, 9000], color=BLACK, lw=1)
axes[0].set_xlabel("MonthlyCharges × tenure")
axes[0].set_ylabel("TotalCharges")
axes[0].set_title("Сумма почти равна платежу, умноженному на срок")

ratio = telco["TotalCharges"] / estimate
axes[1].hist(ratio.dropna(), bins=60, color=BLUE)
axes[1].set_xlabel("TotalCharges / (MonthlyCharges × tenure)")
axes[1].set_title("Отклонения — это смена тарифа за время жизни")
plt.tight_layout()
plt.show()

In [ ]:
telco["TotalCharges"] = telco["TotalCharges"].fillna(0.0)
print("пропусков после заполнения:", telco["TotalCharges"].isna().sum())

### Дубликаты

In [ ]:
duplicated = telco[telco.duplicated(keep=False)].sort_values(list(telco.columns))
print("строк, у которых есть точная копия:", len(duplicated), " лишних копий:", telco.duplicated().sum())
duplicated.head(6)[["gender", "tenure", "PhoneService", "InternetService", "Contract", "MonthlyCharges", "Churn"]]

Первые строки после сортировки случайно оказались с оптоволокном. Посмотрим
на все строки-копии сразу: какой у них интернет, договор и срок.

In [ ]:
display(duplicated["InternetService"].value_counts())
display(duplicated[["Contract", "tenure"]].value_counts())

Копии есть, но это почти наверняка разные клиенты: у всех копий `tenure = 1`,
у большинства только телефон без интернета и один и тот же стартовый тариф,
а таких новичков действительно много. Без идентификатора клиента отличить
настоящий дубль от совпадения нельзя, поэтому строки оставляем. Правило:
дубликат — это гипотеза, которую надо проверить, а не повод удалять.

## 4. Распределения и выбросы

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, column in zip(axes, ["tenure", "MonthlyCharges", "TotalCharges"]):
    ax.hist(telco[column], bins=40, color=BLUE)
    ax.set_title(column)
plt.tight_layout()
plt.show()

Ни одно распределение не похоже на колокол. У `tenure` два пика: новые
клиенты и те, кто с оператором почти с самого начала. У `MonthlyCharges`
большая группа около 20 — клиенты только с телефоном. `TotalCharges`
с тяжелым правым хвостом: большие суммы бывают, но редко.

Хвост еще не значит выбросы. Классический способ их искать — межквартильный
размах. Пусть $Q_1$ и $Q_3$ — первый и третий квартили, $\mathrm{IQR} = Q_3 - Q_1$.
Выбросом называют значение за пределами

$$[\,Q_1 - k \cdot \mathrm{IQR},\ Q_3 + k \cdot \mathrm{IQR}\,], \qquad k = 1.5$$

Это те самые усы на ящике с усами. Функция возвращает маску выбросов:
`True` там, где значение за границами. Пропуски выбросами не считаются.

In [ ]:
def iqr_outliers(values: pd.Series, k: float = 1.5) -> pd.Series:
    q1, q3 = values.quantile(0.25), values.quantile(0.75)
    spread = q3 - q1
    mask = (values < q1 - k * spread) | (values > q3 + k * spread)
    return mask

Ящик с усами рисует ровно эти границы: ящик от $Q_1$ до $Q_3$, усы
до последних значений внутри границ, отдельные точки за ними — выбросы.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
for ax, column in zip(axes, ["tenure", "MonthlyCharges", "TotalCharges"]):
    ax.boxplot(telco[column], orientation="horizontal", whis=1.5, widths=0.6,
               boxprops={"color": BLUE}, medianprops={"color": OCHRE, "lw": 2})
    ax.set_yticks([])
    ax.set_title(f"{column}: выбросов при k = 1.5 — {iqr_outliers(telco[column]).sum()}")
plt.tight_layout()
plt.show()

Выбросов по IQR нет ни одного: хвост `TotalCharges` длинный, но плавный.
Граница зависит от $k$. Посмотрим, сколько «выбросов» получается, если
делать ее все уже.

In [ ]:
k_grid = np.round(np.arange(0.1, 1.55, 0.05), 2)
plt.figure(figsize=(8, 3.8))
for column, color in zip(["tenure", "MonthlyCharges", "TotalCharges"], [BLUE, OCHRE, BLACK]):
    outlier_counts = [iqr_outliers(telco[column], k=k).sum() for k in k_grid]
    plt.plot(k_grid, outlier_counts, "o-", ms=3, color=color, label=column)
plt.axvline(1.5, color=GREY, ls="--", lw=1)
plt.xlabel("k")
plt.ylabel("число выбросов")
plt.title("Чем уже граница, тем больше «выбросов»")
plt.legend()
plt.show()

При $k = 1.5$ выбросов ноль, при $k = 0.5$ у `TotalCharges` их уже почти
тысяча. Попробуйте в ячейке с ящиками поставить `whis=0.5` и посмотрите,
как появятся отдельные точки. Выброс — это значение, которое не вписывается в процесс, породивший
данные: ошибка ввода, другая единица измерения, тестовая запись. Метод IQR
только подсказывает, куда смотреть, решение принимает человек.

## 5. Связь признаков с ответом

Самый прямой способ увидеть, что влияет на отток, — посчитать **долю
ушедших** в каждой категории признака. Берем всех клиентов с этим значением
признака и делим число ушедших среди них на их общее число. Сначала
переведем ответ в нули и единицы.

In [ ]:
telco["churn"] = (telco["Churn"] == "Yes").astype(int)
overall_churn = telco["churn"].mean()
print(f"доля ушедших среди всех клиентов: {overall_churn:.3f}")

Для типа договора считаем три числа на категорию: сколько ушло (сумма
единиц), сколько всего клиентов и их отношение. Среднее нулей и единиц
и есть доля единиц.

In [ ]:
contract_churn = telco.groupby("Contract")["churn"].agg(["sum", "size", "mean"])
contract_churn.columns = ["ушли", "всего клиентов", "доля ушедших"]
show_table(contract_churn)

Теперь то же для трех признаков на графике. Длина столбика — доля ушедших
среди клиентов с этим значением, подпись справа — эта доля и сколько
клиентов в категории. Пунктир — доля ушедших среди всех клиентов, 26.5%:
столбик правее пунктира значит, что в этой категории уходят чаще среднего.
Размер группы важен: доля по двадцати клиентам ничего не значит.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 3.4), sharex=True)
for ax, column in zip(axes, ["Contract", "InternetService", "PaymentMethod"]):
    stats = telco.groupby(column)["churn"].agg(["mean", "size"]).sort_values("mean")
    ax.barh(stats.index, stats["mean"], color=BLUE)
    for y, (share, size) in enumerate(zip(stats["mean"], stats["size"])):
        ax.text(share + 0.01, y, f"{share:.0%} из {size}", va="center", fontsize=10,
                bbox={"facecolor": "white", "edgecolor": "none", "pad": 1})
    ax.axvline(overall_churn, color=BLACK, lw=1, ls="--", label=f"все клиенты: {overall_churn:.1%}")
    ax.set_title(column)
    ax.set_xlabel("доля ушедших среди клиентов с этим значением")
    ax.set_xlim(0, 0.6)
    ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

Помесячный договор — 43% оттока, договор на два года — 3%. Оптоволокно
и оплата электронным чеком тоже заметно выше среднего. Но признаки
связаны между собой: может быть, дело не в чеке, а в том, что
электронным чеком платят как раз клиенты на помесячном договоре.

Таблица сопряженности показывает, как признаки распределены вместе.
С `normalize="index"` каждая строка делится на свою сумму: видно, какой
способ оплаты какую долю составляет внутри каждого типа договора.

In [ ]:
show_table(pd.crosstab(telco["Contract"], telco["PaymentMethod"], normalize="index"), digits=2)

Электронным чеком действительно платит почти половина клиентов
на помесячном договоре и мало кто на двухлетнем. Теперь доля ушедших
в каждой паре «договор × способ оплаты» — сводная таблица `pivot_table`,
нарисованная как тепловая карта.

In [ ]:
telco.pivot_table(index="Contract", columns="PaymentMethod", values="churn", aggfunc="mean")

В цифрах часто можно запутаться, да и они редко бывают наглядными. Немного поиграемся с графиком, чтобы наглядно показать зависимости.

In [ ]:
pair_churn = telco.pivot_table(index="Contract", columns="PaymentMethod", values="churn", aggfunc="mean")

fig, ax = plt.subplots(figsize=(11, 3.6))
image = ax.imshow(pair_churn.to_numpy(), cmap="Oranges", vmin=0, vmax=0.6, aspect="auto")
ax.set_xticks(range(pair_churn.shape[1]), [name.replace(" (", "\n(") for name in pair_churn.columns])
ax.set_yticks(range(pair_churn.shape[0]), pair_churn.index)
for i in range(pair_churn.shape[0]):
    for j in range(pair_churn.shape[1]):
        ax.text(j, i, f"{pair_churn.iloc[i, j]:.0%}", ha="center", va="center",
                color="white" if pair_churn.iloc[i, j] > 0.35 else BLACK)
ax.grid(False)
plt.colorbar(image, ax=ax, label="доля ушедших")
ax.set_title("Доля ушедших в каждой паре «договор × способ оплаты»")
plt.tight_layout()
plt.show()

Внутри одного типа договора электронный чек все еще хуже остальных способов,
так что эффект не сводится к договору целиком. Но он заметно меньше, чем
на общем графике. Отсюда правило: сравнение по одному признаку показывает
связь, а не причину.

### Числовые признаки и корреляции

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, column in zip(axes, ["tenure", "MonthlyCharges", "TotalCharges"]):
    for value, color, label in [(0, GREY, "остался"), (1, OCHRE, "ушел")]:
        ax.hist(telco.loc[telco["churn"] == value, column], bins=40, alpha=0.6, color=color, density=True, label=label)
    ax.set_title(column)
axes[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
numeric = telco[["tenure", "MonthlyCharges", "TotalCharges", "churn"]]
print("Пирсон:")
display(show_table(numeric.corr(method="pearson"), digits=2))
print("Спирмен:")
display(show_table(numeric.corr(method="spearman"), digits=2))

Уходят в основном в первые месяцы. Корреляция Пирсона измеряет линейную
связь, Спирмена — монотонную: это корреляция Пирсона между рангами.
Для `tenure` и `TotalCharges` связь монотонная, но не линейная, поэтому
Спирмен выше. Для бинарной целевой переменной корреляция — грубая мера,
но знак и порядок величины она показывает честно.

## 6. Категориальные признаки

Линейная модель умеет работать только с числами. Как превратить
`Contract` в число?

### One-hot

Каждое значение становится отдельным столбцом из нулей и единиц.
Исходный столбец:

In [ ]:
telco["Contract"].head()

`pd.get_dummies` делает по столбцу на каждое значение.

In [ ]:
contract_dummies = pd.get_dummies(telco["Contract"], dtype=int)
contract_dummies.head()

Рядом с исходным столбцом видно, куда встала единица в каждой строке.

In [ ]:
pd.concat([telco["Contract"], contract_dummies], axis=1).head()

В каждой строке ровно одна единица. Проверим на всех строках: сумма
трех столбцов равна 1 у всех семи тысяч клиентов.

In [ ]:
contract_dummies.sum(axis=1).value_counts()

Раз сумма трех столбцов всегда равна 1, она совпадает со столбцом единиц,
который отвечает за свободный член. Столбцы матрицы признаков линейно
зависимы, и $X^\top X$ вырождена: это та же мультиколлинеарность,
что на втором занятии, только точная. Соберем матрицу со свободным членом
и посмотрим на ее ранг.

In [ ]:
ones = np.ones((len(telco), 1))
full = np.column_stack([ones, contract_dummies.to_numpy(dtype=float)])
print("столбцов:", full.shape[1], " ранг:", np.linalg.matrix_rank(full))

Ранг меньше числа столбцов: один столбец выражается через остальные.
Выбросим первую категорию, `drop_first=True`.

In [ ]:
dropped_dummies = pd.get_dummies(telco["Contract"], dtype=float, drop_first=True)
dropped_dummies.head()

In [ ]:
dropped = np.column_stack([ones, dropped_dummies.to_numpy()])
print("столбцов:", dropped.shape[1], " ранг:", np.linalg.matrix_rank(dropped))

Число обусловленности $X^\top X$ показывает, насколько матрица близка
к вырожденной: чем оно больше, тем сильнее шум в данных раздувается
в весах. Шкала на графике логарифмическая.

In [ ]:
conditions = {name: np.linalg.cond(matrix.T @ matrix) for name, matrix in [("все три столбца", full), ("без первого столбца", dropped)]}
for name, value in conditions.items():
    print(f"{name:20s} число обусловленности X^T X: {value:.1e}")

plt.figure(figsize=(7, 3))
plt.barh(list(conditions), list(conditions.values()), color=[OCHRE, BLUE])
plt.xscale("log")
plt.xlabel("число обусловленности, логарифмическая шкала")
plt.show()

С тремя столбцами и свободным членом число обусловленности бесконечно
по смыслу и огромно численно. Выбросив один столбец, получаем честную
матрицу: выброшенная категория становится точкой отсчета, а веса остальных
показывают отличие от нее. С регуляризацией матрица и так обратима, поэтому
`sklearn` по умолчанию все столбцы оставляет.

### Почему не `get_dummies` на тестовом сете отдельно

Если в тестовом сете нет какой-то категории или есть новая, `get_dummies`
на тестовом сете даст другой набор столбцов, и модель получит на вход не то.
Возьмем маленькое обучение и тестовый сет с новой категорией `Weekly`.

In [ ]:
train_part = pd.DataFrame({"Contract": ["Month-to-month", "One year", "Two year"]})
test_part = pd.DataFrame({"Contract": ["One year", "Weekly"]})
display(train_part)
display(test_part)

In [ ]:
pd.get_dummies(train_part)

In [ ]:
pd.get_dummies(test_part)

На обучении три столбца, на тестовом сете два, и второй из них — `Weekly`,
которого модель никогда не видела. `OneHotEncoder` запоминает категории
на обучении.

In [ ]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder.fit(train_part)
encoder.categories_

На тестовом сете он строит ровно те же три столбца, а незнакомое значение
при `handle_unknown="ignore"` превращает в строку нулей.

In [ ]:
test_encoded = encoder.transform(test_part)
pd.DataFrame(test_encoded, columns=encoder.get_feature_names_out(), index=test_part["Contract"])

### Порядковое кодирование

У `Contract` есть порядок: месяц, год, два года. Такой признак можно
закодировать одним числом 0, 1, 2. Модель получит один вес вместо двух
и будет считать, что переход от месяца к году такой же, как от года
к двум. Иногда это разумно, иногда нет, решает смысл признака.

In [ ]:
ordinal = OrdinalEncoder(categories=[["Month-to-month", "One year", "Two year"]])
ordinal.fit(telco[["Contract"]])
ordinal.categories_

Порядок категорий задан явно, и номер каждой — ее место в этом списке.
Что было и что стало:

In [ ]:
contract_ordinal = ordinal.transform(telco[["Contract"]])[:, 0]
pd.DataFrame({"Contract": telco["Contract"], "порядковый код": contract_ordinal}).sample(5)

### Частотное кодирование

Вместо категории — ее доля в обучающих данных. Модель узнает, редкая это
категория или частая, но не узнает, какая именно. Сначала доли.

In [ ]:
payment_share = telco["PaymentMethod"].value_counts(normalize=True)
payment_share

Потом каждая строка получает долю своей категории.

In [ ]:
payment_frequency = telco["PaymentMethod"].map(payment_share)
pd.DataFrame({"PaymentMethod": telco["PaymentMethod"], "частота": payment_frequency.round(3)}).head()

### Target encoding

Для признака с сотнями значений one-hot даст сотни столбцов. Можно заменить
категорию на среднее значение целевой переменной в ней — долю ушедших
клиентов с таким значением. Для редкой категории такое среднее шумное,
поэтому его подтягивают к общему среднему $\bar{y}$.

Пусть $n_c$ — сколько раз категория $c$ встретилась в обучающих данных,
$\bar{y}_c$ — среднее ответа в ней, то есть сумма ответов в категории
равна $n_c \bar{y}_c$. Добавим к категории $m$ воображаемых клиентов,
у каждого из которых ответ равен общему среднему $\bar{y}$. Сумма ответов
станет $n_c \bar{y}_c + m \bar{y}$, клиентов — $n_c + m$, и среднее по ним

$$\mathrm{enc}(c) = \frac{n_c \cdot \bar{y}_c + m \cdot \bar{y}}{n_c + m}.$$

Разделим дробь на два слагаемых:

$$\mathrm{enc}(c) = \frac{n_c}{n_c + m}\,\bar{y}_c + \frac{m}{n_c + m}\,\bar{y} = \lambda_c \bar{y}_c + (1 - \lambda_c)\,\bar{y}, \qquad \lambda_c = \frac{n_c}{n_c + m}.$$

Кодировка — взвешенное среднее двух чисел, $m$ — сила сглаживания. Если $n_c \gg m$,
слагаемыми с $m$ можно пренебречь: $\mathrm{enc}(c) \approx \frac{n_c \bar{y}_c}{n_c} = \bar{y}_c$.
Если $n_c \ll m$, пренебрегаем слагаемыми с $n_c$: $\mathrm{enc}(c) \approx \frac{m \bar{y}}{m} = \bar{y}$.
Это та же идея, что априорное распределение на втором занятии: пока данных
мало, верим общему среднему.

Сумма $n_c \bar{y}_c$ — это просто сумма ответов в категории, поэтому
в коде удобнее считать сумму и число строк. Незнакомая категория
получает $\bar{y}$.

In [ ]:
def target_encode(train_values: pd.Series, train_target: pd.Series, values: pd.Series,
                  smoothing: float = 10.0) -> np.ndarray:
    prior = train_target.mean()
    stats = pd.DataFrame({"value": train_values.to_numpy(), "target": train_target.to_numpy()})
    grouped = stats.groupby("value")["target"].agg(["sum", "count"])
    encoding = (grouped["sum"] + smoothing * prior) / (grouped["count"] + smoothing)
    encoded = values.map(encoding).fillna(prior).to_numpy(dtype=float)
    return encoded

**Что было и что стало** на способе оплаты. Для каждой категории — сколько
в ней клиентов, доля ушедших и кодировка при разной силе сглаживания.

In [ ]:
payment_stats = telco.groupby("PaymentMethod")["churn"].agg(["size", "mean"])
payment_stats.columns = ["n_c", "доля ушедших"]
for m in [0, 10, 1000]:
    payment_stats[f"enc, m = {m}"] = target_encode(telco["PaymentMethod"], telco["churn"], pd.Series(payment_stats.index), smoothing=m)
show_table(payment_stats)

При $m = 0$ кодировка равна доле ушедших. При $m = 10$ она почти не меняется:
в каждой категории больше полутора тысяч клиентов. При $m = 1000$ все
значения заметно подтянуты к общему 0.265. Добавьте в список `m = 100`
и `m = 10000` и посмотрите, при каком $m$ кодировки почти сливаются.
Так выглядят сами строки таблицы до и после:

In [ ]:
pd.DataFrame({
    "PaymentMethod, было": telco["PaymentMethod"],
    "target encoding, стало": target_encode(telco["PaymentMethod"], telco["churn"], telco["PaymentMethod"]).round(3),
}).head(6)

Сглаживание нужно там, где категорий много, а клиентов в каждой мало.
Добавим клиентам признак, который вообще ничего не значит: случайный
«номер офиса» от 0 до 2999. Часть номеров не досталась ни одному клиенту,
поэтому офисов с клиентами чуть меньше трех тысяч. В каждом офисе в среднем два-три
клиента, и настоящая доля ушедших в каждом офисе одна и та же — общая.

In [ ]:
rng_office = np.random.default_rng(0)
office = pd.Series(rng_office.integers(0, 3000, len(telco)).astype(str), index=telco.index)
office_sizes = office.value_counts()
print(f"офисов: {office.nunique()}  клиентов в офисе: медиана {int(office_sizes.median())}, максимум {office_sizes.max()}")

Номер офиса случайный, поэтому клиентов в офисах поровну только в среднем:
где-то один клиент, где-то десять. Сколько офисов каждого размера:

In [ ]:
size_counts = office_sizes.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar(size_counts.index, size_counts.to_numpy(), color=BLUE)
ax.set_xticks(size_counts.index)
ax.set_xlabel("клиентов в офисе")
ax.set_ylabel("офисов")
plt.tight_layout()
plt.show()

Закодируем офисы по всем клиентам и посмотрим, какие кодировки получились.
В офисе из $n$ клиентов могут уйти $0, 1, \dots, n$ человек, поэтому
без сглаживания у кодировки всего $n + 1$ возможное значение:
$0, \frac{1}{n}, \frac{2}{n}, \dots, 1$. На графике по горизонтали — сколько
клиентов в офисе, по вертикали — кодировка. Каждый кружок — одно возможное
значение, его площадь — сколько офисов получили такую кодировку.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for ax, m in zip(axes, [0, 10]):
    encoded = target_encode(office, telco["churn"], pd.Series(office_sizes.index), smoothing=m)
    offices = pd.DataFrame({"клиентов": office_sizes.to_numpy(), "кодировка": encoded.round(6)})
    bubbles = offices.value_counts().rename("офисов").reset_index()
    ax.scatter(bubbles["клиентов"], bubbles["кодировка"], s=10 + 1.5 * bubbles["офисов"], color=BLUE, alpha=0.6, edgecolor="white")
    ax.axhline(overall_churn, color=BLACK, ls="--", lw=1, label="общая доля ушедших")
    ax.set_xticks(range(1, office_sizes.max() + 1))
    ax.set_xlabel("клиентов в офисе")
    ax.set_title(f"m = {m}")
for count in [10, 100, 500]:
    axes[1].scatter([], [], s=10 + 1.5 * count, color=BLUE, alpha=0.6, label=f"{count} офисов")
axes[0].set_ylabel("кодировка офиса")
axes[1].legend(loc="upper right", labelspacing=1.3)
plt.tight_layout()
plt.show()

При $m = 0$ маленькие офисы получают 0 или 1: у офиса с одним клиентом
кодировка — это просто его ответ. Такой разброс — чистый шум: настоящая доля
ушедших во всех офисах одна и та же. При $m = 10$ к каждому офису добавлены
десять воображаемых клиентов с общей долей ушедших $\bar{y} \approx 0.265$,
и все кодировки подтянуты к пунктиру. Офис из одного ушедшего клиента получает

$$\mathrm{enc}(c) = \frac{n_c \cdot \bar{y}_c + m \cdot \bar{y}}{n_c + m} = \frac{1 \cdot 1 + 10 \cdot 0.265}{1 + 10} = \frac{3.65}{11} \approx 0.33,$$

а офис из одного оставшегося —

$$\mathrm{enc}(c) = \frac{1 \cdot 0 + 10 \cdot 0.265}{1 + 10} = \frac{2.65}{11} \approx 0.24.$$

Это два кружка при одном клиенте на правом графике.

### Утечка в target encoding

У target encoding есть ловушка, и номер офиса ее хорошо показывает.
Как всегда, сначала разобьем клиентов на обучение и тестовый сет. Кодировку
будем считать по обучению, а смотреть, как закодированный признак связан
с ответом, — и на обучении, и на тестовом сете. Мера связи — корреляция закодированного
признака с ответом: около нуля — связи нет, чем ближе к единице, тем сильнее
признак подсказывает ответ. У номера офиса настоящей связи с уходом нет,
поэтому честная кодировка должна дать около нуля и там, и там.

In [ ]:
office_train, office_test, y_office_train, y_office_test = train_test_split(
    office, telco["churn"], test_size=0.25, random_state=SEED, stratify=telco["churn"]
)
print("обучение:", len(office_train), " тестовый сет:", len(office_test))

**Наивная кодировка.** Так будем называть самый прямой способ: посчитать
по обучению долю ушедших в каждом офисе и этими же числами закодировать
те же строки обучения. В коде это `target_encode(office_train, y_office_train, office_train)`:
и доли, и кодируемые строки берутся из обучения. Сглаживание выключим,
$m = 0$, так эффект виден лучше всего.

Как читать таблицу. Слева индекс — номер клиента, то есть номер его строки
в исходной таблице `telco`. `train_test_split` перемешал клиентов, поэтому
номера идут вразнобой, но за каждым номером тот же человек, что и в `telco`.
Номер клиента и номер офиса — разные числа: во второй строке клиент 4811
из офиса 2841. «офис, было» — значение признака до кодирования, «ушел» —
ответ, «кодировка, стало» — число, которым заменили номер офиса: доля
ушедших среди клиентов обучения из того же офиса.

In [ ]:
enc_train_naive = target_encode(office_train, y_office_train, office_train, smoothing=0)
pd.DataFrame({"офис, было": office_train, "ушел": y_office_train, "кодировка, стало": enc_train_naive.round(3)}).rename_axis("клиент").head(8)

Откуда у клиента 4811 число 0.667. Выпишем всех клиентов офиса 2841,
и из обучения, и из тестового сета.

In [ ]:
example_office = "2841"
example_clients = office.index[office == example_office]
example = pd.DataFrame({
    "часть": np.where(example_clients.isin(office_train.index), "обучение", "тестовый сет"),
    "офис": example_office,
    "ушел": telco.loc[example_clients, "churn"],
}, index=example_clients).rename_axis("клиент")
example.sort_values("часть", kind="stable")

В обучении у офиса три клиента, двое из них ушли. Наивная кодировка офиса —
доля ушедших среди них:

In [ ]:
example_answers = y_office_train[office_train == example_office]
example_encoding = target_encode(office_train, y_office_train, pd.Series([example_office]), smoothing=0)[0]
print("ответы клиентов обучения:", example_answers.tolist())
print(f"кодировка офиса: {example_answers.sum()} / {len(example_answers)} = {example_encoding:.3f}")

Число 0.667 получают все три клиента обучения, и в нем сидят их собственные
ответы: двое ушедших сами подняли долю ушедших в своем офисе. Клиент тестового сета
1250 получит то же 0.667, но его ответ в эту долю не входил: она посчитана
по другим людям. Номер офиса случаен, поэтому ответы соседей по офису
про него ничего не говорят.

Крайний случай — офис, в котором в обучении один клиент, как офис 2255
в первой строке. Его кодировка — доля ушедших среди одного человека,
то есть в точности его ответ, 0 или 1. Посчитаем, как часто кодировка
совпадает с собственным ответом строки:

In [ ]:
train_office_sizes = office_train.value_counts()
same_as_answer = np.mean(enc_train_naive == y_office_train.to_numpy())
print("офисов с одним клиентом в обучении:", (train_office_sizes == 1).sum(), "из", len(train_office_sizes))
print(f"строк обучения, у которых кодировка равна их собственному ответу: {same_as_answer:.0%}")

Больше чем у половины строк обучения вместо номера офиса стоит их собственный
ответ: это офисы с одним клиентом и офисы, где все клиенты обучения ответили
одинаково. Для модели это готовая подсказка.

На тестовом сете кодировка берется из тех же долей по обучению. Ответ тестового
клиента в них не участвовал, и подсказки нет:

In [ ]:
enc_test_naive = target_encode(office_train, y_office_train, office_test, smoothing=0)
pd.DataFrame({"офис, было": office_test, "ушел": y_office_test, "кодировка, стало": enc_test_naive.round(3)}).rename_axis("клиент").head(8)

Теперь связь с ответом на обучении и на тестовом сете:

In [ ]:
print(f"наивная кодировка: корреляция с ответом на обучении {np.corrcoef(enc_train_naive, y_office_train)[0, 1]:.3f}, "
      f"на тестовом сете {np.corrcoef(enc_test_naive, y_office_test)[0, 1]:.3f}")

Откуда берется корреляция, видно по гистограммам кодировки, построенным
отдельно для оставшихся и для ушедших. Если признак ничего не говорит
об ответе, две гистограммы совпадают. Чем сильнее они разъезжаются, тем
больше корреляция.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), sharey=True)
for ax, (title, encoded, labels) in zip(axes, [
    ("наивная кодировка на обучении", enc_train_naive, y_office_train),
    ("та же кодировка на тестовом сете", enc_test_naive, y_office_test),
]):
    for value, color, label in [(0, GREY, "остался"), (1, OCHRE, "ушел")]:
        ax.hist(encoded[labels.to_numpy() == value], bins=20, range=(0, 1), alpha=0.6, density=True, color=color, label=label)
    ax.set_title(title)
    ax.set_xlabel("кодировка номера офиса")
axes[0].legend()
plt.tight_layout()
plt.show()

На обучении у оставшихся кодировка чаще всего 0 и никогда не бывает 1,
а у ушедших, наоборот, никогда не бывает 0: собственный ответ строки входит
в долю ее офиса и тянет кодировку к себе. Признак почти повторяет ответ.
На тестовом сете гистограммы одинаковые.
Кодировка тестового клиента посчитана по чужим ответам, и видна настоящая
связь номера офиса с уходом — нулевая. Поэтому на тестовом сете проблемы и не видно:
тестовый сет закодирован честно, а подсказка сидит только в строках обучения.
Модель, обученная на таком признаке, поверит ему больше всех остальных
и на новых данных провалится.

### Кросс-фиттинг

Лечение — кросс-фиттинг. Делим обучающую выборку на $K$ частей, фолдов.
Кодировку для строк каждого фолда считаем по остальным $K - 1$ фолдам.
Так ответ строки никогда не участвует в ее собственном признаке: строка
обучения кодируется так же честно, как строка тестового сета. Сделаем это руками,
$K = 5$, и запомним, в какой фолд попала каждая строка.

In [ ]:
office_folds = KFold(5, shuffle=True, random_state=SEED)
enc_train_cross = np.zeros(len(office_train))
fold_of_row = np.zeros(len(office_train), dtype=int)
for fold, (fit_rows, encode_rows) in enumerate(office_folds.split(office_train)):
    fold_encoding = target_encode(office_train.iloc[fit_rows], y_office_train.iloc[fit_rows], office_train.iloc[encode_rows], smoothing=0)
    enc_train_cross[encode_rows] = fold_encoding
    fold_of_row[encode_rows] = fold

Тот же офис 2841: в каком фолде каждый клиент обучения и чем он закодирован
обоими способами.

In [ ]:
in_example = (office_train == example_office).to_numpy()
pd.DataFrame({
    "фолд": fold_of_row[in_example],
    "ушел": y_office_train[in_example],
    "наивно": enc_train_naive[in_example].round(3),
    "кросс-фиттинг": enc_train_cross[in_example].round(3),
}, index=office_train.index[in_example]).rename_axis("клиент")

Клиенты 4811 и 4074 попали в один фолд. Их кодировка посчитана по остальным
фолдам, а там из офиса 2841 есть только клиент 2835, и он остался:
кодировка 0. Клиента 2835 кодируют по двум другим, оба ушли: кодировка 1.
Наивно все трое получали 0.667, и туда входили их собственные ответы. После
кросс-фиттинга кодировка каждого — это ответы соседей по офису, про его
собственный ответ она ничего не знает. Если в других фолдах у офиса никого
нет, про офис ничего не известно, и строка получает общую долю ушедших
по этим фолдам, около 0.265.

`TargetEncoder.fit_transform` в sklearn делает ровно это. По умолчанию
он еще и сглаживает; `smooth=0` выключает сглаживание, чтобы от наивной
кодировки нас отличал только кросс-фиттинг. `cv=office_folds` дает ему
те же пять фолдов, что в нашем цикле. Без этого `TargetEncoder` сам режет
обучение на пять частей с перемешиванием и без `random_state`: фолды были бы
другими, и числа совпали бы с циклом только по смыслу, а не поштучно.
Сверим с нашим циклом:

In [ ]:
office_encoder = TargetEncoder(smooth=0, cv=office_folds)
enc_train_sklearn = office_encoder.fit_transform(office_train.to_frame(), y_office_train)[:, 0]
print("sklearn совпадает с циклом:", np.allclose(enc_train_sklearn, enc_train_cross))

Было и стало на тех же восьми клиентах, что в первой таблице:

In [ ]:
pd.DataFrame({
    "офис": office_train,
    "ушел": y_office_train,
    "наивно": enc_train_naive.round(3),
    "кросс-фиттинг": enc_train_cross.round(3),
}).rename_axis("клиент").head(8)

Наивная кодировка повторяла столбец «ушел», кросс-фиттинг — нет. Клиент 6661
единственный из офиса 2255 в обучении, и вместо его собственного ответа
он теперь получает общую долю ушедших.

На тестовом сете кросс-фиттинг ничего не меняет: `transform` кодирует новые строки
долями по всему обучению, как и наивная кодировка.

In [ ]:
enc_test_cross = office_encoder.transform(office_test.to_frame())[:, 0]
print("на тестовом сете совпадает с наивной кодировкой:", np.allclose(enc_test_cross, enc_test_naive))

In [ ]:
print(f"кросс-фиттинг: корреляция с ответом на обучении {np.corrcoef(enc_train_cross, y_office_train)[0, 1]:.3f}, "
      f"на тестовом сете {np.corrcoef(enc_test_cross, y_office_test)[0, 1]:.3f}")

Было и стало на одной картинке: сверху наивная кодировка, снизу
кросс-фиттинг, слева обучение, справа тестовый сет.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 6.5), sharex=True)
panels = [
    ("наивно, обучение", enc_train_naive, y_office_train),
    ("наивно, тестовый сет", enc_test_naive, y_office_test),
    ("кросс-фиттинг, обучение", enc_train_cross, y_office_train),
    ("кросс-фиттинг, тестовый сет", enc_test_cross, y_office_test),
]
for ax, (title, encoded, labels) in zip(axes.ravel(), panels):
    for value, color, label in [(0, GREY, "остался"), (1, OCHRE, "ушел")]:
        ax.hist(encoded[labels.to_numpy() == value], bins=20, range=(0, 1), alpha=0.6, density=True, color=color, label=label)
    ax.set_title(title)
for ax in axes[1]:
    ax.set_xlabel("кодировка номера офиса")
axes[0, 0].legend()
plt.tight_layout()
plt.show()

Верхний ряд — то, что было: на обучении гистограммы разъехались, на тестовом сете
совпадают. Нижний ряд — кросс-фиттинг: обучение выглядит так же, как тестовый сет,
и признак честно показывает, что про уход он ничего не знает. Модель
не станет на него опираться.

Высокий столбик около 0.27 есть на всех панелях, кроме верхней левой. Это
строки, про офис которых кодировщик ничего не знает, и они получают общую
долю ушедших. На тестовом сете это клиенты из офисов, которых нет в обучении.
Внизу слева столбик выше: каждый фолд кодируется по четырем пятым обучения,
и незнакомых офисов там больше. Посчитаем обе доли:

In [ ]:
unknown_in_test = np.mean(~office_test.isin(office_train))
prior_in_cross = np.mean(np.abs(enc_train_cross - y_office_train.mean()) < 0.01)
print(f"строк тестового сета из офисов, которых нет в обучении: {unknown_in_test:.0%}")
print(f"строк обучения, получивших при кросс-фиттинге общую долю: {prior_in_cross:.0%}")

**Как ведет себя настоящий признак.** Сравним номер офиса со способом
оплаты, который действительно связан с уходом, на том же разбиении.
Функция ниже делает для любого признака то же, что мы проделали с офисом:
наивная кодировка и кросс-фиттинг, на обучении и на тестовом сете. Результат —
две таблицы, обучение и тестовый сет, по столбцу на способ кодирования.

In [ ]:
def encode_two_ways(values: pd.Series, train_index, test_index) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_values, test_values = values.loc[train_index], values.loc[test_index]
    y_train_part = telco.loc[train_index, "churn"]
    cross_encoder = TargetEncoder(smooth=0, cv=KFold(5, shuffle=True, random_state=SEED))
    train_codes = pd.DataFrame({
        "наивно": target_encode(train_values, y_train_part, train_values, smoothing=0),
        "кросс-фиттинг": cross_encoder.fit_transform(train_values.to_frame(), y_train_part)[:, 0],
    }, index=train_index)
    test_codes = pd.DataFrame({
        "наивно": target_encode(train_values, y_train_part, test_values, smoothing=0),
        "кросс-фиттинг": cross_encoder.transform(test_values.to_frame())[:, 0],
    }, index=test_index)
    return train_codes, test_codes

In [ ]:
office_train_codes, office_test_codes = encode_two_ways(office, office_train.index, office_test.index)
payment_train_codes, payment_test_codes = encode_two_ways(telco["PaymentMethod"], office_train.index, office_test.index)

Сложим ответ и все четыре кодировки в одну таблицу, отдельно для обучения
и для тестового сета.

In [ ]:
codes_train = pd.concat([y_office_train.rename("ушел"), office_train_codes.add_prefix("офис, "), payment_train_codes.add_prefix("оплата, ")], axis=1)
codes_test = pd.concat([y_office_test.rename("ушел"), office_test_codes.add_prefix("офис, "), payment_test_codes.add_prefix("оплата, ")], axis=1)
codes_train.rename_axis("клиент").head().round(3)

Матрица корреляций: в клетке — корреляция двух столбцов, слева на обучении,
справа на тестовом сете. Шкала общая, от −1 до 1: синий — отрицательная связь,
красный — положительная, белый — никакой.

In [ ]:
corr_train = codes_train.corr()
corr_test = codes_test.corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5.6), layout="constrained")
for ax, (title, corr) in zip(axes, [("обучение", corr_train), ("тестовый сет", corr_test)]):
    image = ax.imshow(corr.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
    for i in range(len(corr)):
        for j in range(len(corr)):
            value = corr.iloc[i, j]
            ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=10, color="white" if abs(value) > 0.6 else BLACK)
    ax.set_xticks(range(len(corr)), [name.replace(", ", "\n") for name in corr.columns])
    ax.set_yticks(range(len(corr)), corr.index)
    ax.grid(False)
    ax.set_title(title)
fig.colorbar(image, ax=axes, shrink=0.8)
plt.show()

Как читать матрицу. Первая строка — связь каждой кодировки с ответом,
ради нее все и затевалось. У способа оплаты она одинаковая на обучении
и на тестовом сете при любом способе кодирования. У номера офиса наивная кодировка
на обучении дает сильную связь, а на тестовом сете ее нет.

Остальные клетки — связь кодировок между собой. У способа оплаты наивная
кодировка и кросс-фиттинг почти одно и то же, корреляция 1.00: в каждой
категории больше тысячи клиентов обучения, и ответ одного клиента почти
не сдвигает долю. У номера офиса на обучении это разные признаки,
корреляция между ними всего 0.55. Наивная кодировка офиса на обучении
даже коррелирует со способом оплаты, 0.20, хотя офис случаен: она несет
в себе ответ, а ответ связан с оплатой. На тестовом сете обе кодировки считаются
одинаково, по долям всего обучения, и совпадают.

Вынесем первую строку матриц отдельно: корреляция каждой кодировки с ответом
на обучении и на тестовом сете. Одна шкала цвета на всю таблицу, так сразу видно,
какая клетка выбивается.

In [ ]:
answer_corr = pd.DataFrame({"обучение": corr_train["ушел"], "тестовый сет": corr_test["ушел"]}).drop(index="ушел")
show_table(answer_corr, axis=None)

Те же числа столбиками. Посмотрите сначала на настоящий признак, способ
оплаты, — две правые пары столбиков. Так коррелирует с ответом нормальный
признак: связь заметная, и на обучении она такая же, как на тестовом сете, при любом
способе кодирования. Теперь номер офиса, две левые пары. При наивной
кодировке столбик обучения высокий, а столбик тестового сета около нуля: на обучении
признак подсмотрел ответ. С кросс-фиттингом оба около нуля, как и должно
быть у шума.

In [ ]:
positions = np.arange(len(answer_corr))
fig, ax = plt.subplots(figsize=(10, 4))
train_bars = ax.bar(positions - 0.2, answer_corr["обучение"], width=0.4, color=BLUE, label="обучение")
test_bars = ax.bar(positions + 0.2, answer_corr["тестовый сет"], width=0.4, color=OCHRE, label="тестовый сет")
ax.bar_label(train_bars, fmt="%.2f", fontsize=8, padding=2)
ax.bar_label(test_bars, fmt="%.2f", fontsize=8, padding=2)
ax.set_xticks(positions, [name.replace(", ", "\n") for name in answer_corr.index])
ax.axhline(0, color=BLACK, lw=0.8)
ax.set_ylabel("корреляция кодировки с ответом")
ax.legend()
plt.tight_layout()
plt.show()

### Что делать с обучением и тестовым сетом

1. Сначала разбить данные на обучение и тестовый сет и только потом считать
   что-либо по ответам. Любая статистика по целевой переменной — доля ушедших
   в категории, среднее по группе — считается только по обучению.
2. Строки обучения кодировать кросс-фиттингом: `encoder.fit_transform(X_train, y_train)`.
   Не `encoder.fit(X_train, y_train).transform(X_train)`: это и есть наивная
   кодировка, у `TargetEncoder` эти два вызова дают разный результат.
3. Тестовый сет и любые новые данные кодировать `encoder.transform(X_test)`, долями
   по всему обучению. Ответы тестового сета не участвуют ни в кодировке, ни в чем-либо
   еще до финальной оценки модели.
4. Кодировщик класть в `Pipeline` вместе с моделью. `Pipeline.fit` вызывает
   у кодировщика `fit_transform`, то есть кросс-фиттинг, а `predict` —
   `transform`. При кросс-валидации кодировщик тогда обучается заново
   на каждом разбиении и не видит ответов валидационной части. Подробнее —
   в разделе про утечку через предобработку.
5. Проверять себя: если признак на обучении связан с ответом заметно сильнее,
   чем на тестовом сете или на валидационной части, он, скорее всего, подсмотрел ответ.

## 7. Метрики классификации

### Модель по частям

Соберем модель: числовые признаки масштабируем, категориальные кодируем
one-hot, сверху логистическая регрессия. Сначала данные и разбиение.

In [ ]:
feature_frame = telco.drop(columns=["Churn", "churn"])
target = telco["churn"]
numeric_columns = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_columns = [c for c in feature_frame.columns if c not in numeric_columns]

X_train, X_test, y_train, y_test = train_test_split(
    feature_frame, target, test_size=0.25, random_state=SEED, stratify=target
)
print("обучение:", X_train.shape, " тестовый сет:", X_test.shape, f" доля ушедших в тестовом сете {y_test.mean():.3f}")

Обработка числовых столбцов — два шага подряд: заполнить пропуски медианой
и стандартизировать. `make_pipeline` склеивает их в один объект,
`fit_transform` обучает оба шага на обучении и сразу применяет. В нашей
таблице пропусков уже нет: пустые `TotalCharges` мы заменили нулем
в разделе про чистку. Первый шаг здесь ничего не меняет, он страхует
от пропусков в новых данных.

In [ ]:
numeric_steps = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
numeric_prepared = numeric_steps.fit_transform(X_train[numeric_columns])
print("столбцов было:", len(numeric_columns), " стало:", numeric_prepared.shape[1])
pd.DataFrame(numeric_prepared, columns=numeric_steps.get_feature_names_out()).head(3).round(2)

Категориальные столбцы — one-hot, как в предыдущем разделе. Шаг здесь один,
но и его оборачиваем в `make_pipeline`: обе группы столбцов устроены
одинаково, и в любую легко добавить шаг. `sparse_output=False` просит
обычный массив вместо разреженной матрицы, чтобы показать его таблицей.

In [ ]:
categorical_steps = make_pipeline(OneHotEncoder(handle_unknown="ignore", sparse_output=False))
categorical_prepared = categorical_steps.fit_transform(X_train[categorical_columns])
print("столбцов было:", len(categorical_columns), " стало:", categorical_prepared.shape[1])
pd.DataFrame(categorical_prepared, columns=categorical_steps.get_feature_names_out()).head(3).round(2)

`ColumnTransformer` применяет к каждой группе столбцов свое преобразование
и склеивает результаты в одну матрицу.

In [ ]:
preprocess = ColumnTransformer([
    ("numeric", numeric_steps, numeric_columns),
    ("categorical", categorical_steps, categorical_columns),
])
prepared = preprocess.fit_transform(X_train)
print("матрица признаков:", prepared.shape)

И наконец, модель поверх предобработки. Такой `Pipeline` умеет `fit`
и `predict` прямо на сырой таблице.

In [ ]:
logistic_model = make_pipeline(preprocess, LogisticRegression(max_iter=2000))
logistic_model.fit(X_train, y_train)
logistic_model.predict_proba(X_test)[:5].round(3)

Моделей будет много, поэтому сложим те же строки в функцию. Внутри она
делает ровно то, что выше, но при каждом вызове собирает новые объекты:
иначе разные модели делили бы один и тот же обученный `StandardScaler`.

In [ ]:
def make_model(classifier):
    numeric_steps = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    categorical_steps = make_pipeline(OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    preprocess = ColumnTransformer([
        ("numeric", numeric_steps, numeric_columns),
        ("categorical", categorical_steps, categorical_columns),
    ])
    model = make_pipeline(preprocess, classifier)
    return model

### Матрица ошибок

Модель выдает вероятность ухода, ответ получается порогом: 1, если
вероятность не меньше 0.5. Положительный класс — «ушел». Все исходы
на тестовом сете раскладываются по четырем клеткам:

| | предсказано «ушел» | предсказано «остался» |
|---|---|---|
| **на самом деле ушел** | $TP$, верно найденные | $FN$, пропущенные |
| **на самом деле остался** | $FP$, ложные тревоги | $TN$, верно отклоненные |

Всего объектов $n = TP + FP + FN + TN$.

In [ ]:
proba_logistic = logistic_model.predict_proba(X_test)[:, 1]
predicted_logistic = (proba_logistic >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, predicted_logistic).ravel()

cells_layout = np.array([[tp, fn], [fp, tn]])
names = np.array([["TP", "FN"], ["FP", "TN"]])
fig, ax = plt.subplots(figsize=(5.5, 4.2))
ax.imshow(cells_layout, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{names[i, j]}\n{cells_layout[i, j]}", ha="center", va="center", fontsize=13,
                color="white" if cells_layout[i, j] > cells_layout.max() / 2 else BLACK)
ax.set_xticks([0, 1], ["предсказано «ушел»", "предсказано «остался»"])
ax.set_yticks([0, 1], ["ушел", "остался"])
ax.grid(False)
ax.set_title("Логистическая регрессия, порог 0.5")
plt.tight_layout()
plt.show()

### Метрики по меткам

**Accuracy** — доля верных ответов:

$$\mathrm{accuracy} = \frac{TP + TN}{n}$$

**Precision** — из тех, кого модель назвала ушедшими, какая доля ушла
на самом деле:

$$P = \frac{TP}{TP + FP}$$

**Recall** — из всех ушедших какую долю модель нашла. Другое имя — TPR,
доля верно найденных положительных:

$$R = \frac{TP}{TP + FN}$$

**F1** — среднее гармоническое precision и recall:

$$F_1 = \frac{2}{\frac{1}{P} + \frac{1}{R}}$$

Приведем знаменатель к общему знаменателю: $\frac{1}{P} + \frac{1}{R} = \frac{R + P}{P R}$,
поэтому

$$F_1 = \frac{2 P R}{P + R}.$$

Подставим сюда $P = \frac{TP}{TP + FP}$ и $R = \frac{TP}{TP + FN}$. Числитель:

$$2 P R = \frac{2\,TP^2}{(TP + FP)(TP + FN)}.$$

Знаменатель, приведенный к тому же общему знаменателю:

$$P + R = \frac{TP\,(TP + FN) + TP\,(TP + FP)}{(TP + FP)(TP + FN)} = \frac{TP\,(2\,TP + FP + FN)}{(TP + FP)(TP + FN)}.$$

Делим одно на другое, произведения $(TP + FP)(TP + FN)$ сокращаются:

$$F_1 = \frac{2\,TP^2}{TP\,(2\,TP + FP + FN)} = \frac{2\,TP}{2\,TP + FP + FN}.$$

Разделим числитель и знаменатель на 2:

$$F_1 = \frac{2\,TP / 2}{(2\,TP + FP + FN) / 2} = \frac{TP}{TP + \frac{1}{2}(FP + FN)}.$$

В такой записи видно, чем F1 хороша. В числителе верно найденные ушедшие,
в знаменателе к ним добавлена половина всех ошибок модели: ложных тревог $FP$,
то есть оставшихся, названных ушедшими, и пропусков $FN$, то есть ушедших,
названных оставшимися. Обе ошибки штрафуются одинаково,
и F1 близка к единице, только когда мало и тех, и других. Считается она
по положительному классу, поэтому при дисбалансе не обманывает, как accuracy.
Модель, которая говорит «никто не уйдет», получает accuracy около 0.73 —
это доля оставшихся, — но не находит ни одного ушедшего: $TP = 0$, и F1
равна нулю.

Посчитаем все четыре метрики руками по формулам и сверим с sklearn.

In [ ]:
by_hand = {
    "accuracy": (tp + tn) / (tp + fp + fn + tn),
    "precision": tp / (tp + fp),
    "recall": tp / (tp + fn),
    "F1": 2 * tp / (2 * tp + fp + fn),
}
by_sklearn = {
    "accuracy": accuracy_score(y_test, predicted_logistic),
    "precision": precision_score(y_test, predicted_logistic),
    "recall": recall_score(y_test, predicted_logistic),
    "F1": f1_score(y_test, predicted_logistic),
}
show_table(pd.DataFrame({"руками": by_hand, "sklearn": by_sklearn}))

Почему гармоническое, а не обычное среднее. Пусть модель находит 90%
ушедших, recall = 0.9, а precision мы меняем от 0 до 1. Обычное среднее
при нулевой precision все еще 0.45 и выглядит прилично. Гармоническое
падает к нулю: F1 высокий, только если высоки обе метрики сразу.

In [ ]:
precision_grid = np.linspace(0.001, 1, 200)
recall_fixed = 0.9
plt.figure(figsize=(7, 3.8))
plt.plot(precision_grid, (precision_grid + recall_fixed) / 2, color=GREY, lw=2, label="обычное среднее")
plt.plot(precision_grid, 2 * precision_grid * recall_fixed / (precision_grid + recall_fixed), color=BLUE, lw=2, label="F1, гармоническое")
plt.xlabel("precision при recall = 0.9")
plt.ylabel("среднее")
plt.legend()
plt.show()

### Метрики по вероятностям: ROC-AUC и PR-AUC

Метрики выше зависят от порога. Будем двигать порог $t$ от 1 до 0 и для
каждого считать две доли:

$$\mathrm{TPR}(t) = \frac{TP(t)}{TP(t) + FN(t)}, \qquad \mathrm{FPR}(t) = \frac{FP(t)}{FP(t) + TN(t)}.$$

Для сравнения — precision и recall из прошлого раздела при том же пороге:

$$P(t) = \frac{TP(t)}{TP(t) + FP(t)}, \qquad R(t) = \frac{TP(t)}{TP(t) + FN(t)}.$$

Recall и TPR — одна и та же формула. Разница в знаменателях:

| метрика | числитель | знаменатель | кто в знаменателе |
|---|---|---|---|
| TPR, он же recall | $TP(t)$ | $TP(t) + FN(t)$ | все ушедшие |
| FPR | $FP(t)$ | $FP(t) + TN(t)$ | все оставшиеся |
| precision | $TP(t)$ | $TP(t) + FP(t)$ | все, кого модель назвала ушедшими |

TPR и FPR делят на целый класс, precision — на тех, кого выбрала модель.

Все четыре клетки здесь зависят от порога. Модель называет объект ушедшим,
если его оценка не меньше $t$, и $TP(t)$, $FP(t)$, $FN(t)$, $TN(t)$ — клетки
матрицы ошибок при этом пороге. Опускаем порог — объекты с оценкой между
старым и новым порогом переходят из «предсказано остался» в «предсказано
ушел»: ушедшие из $FN$ в $TP$, оставшиеся из $TN$ в $FP$.

Посмотрим на это глазами. Возьмем случайно десять ушедших и десять
оставшихся клиентов тестового сета и их оценки от логистической регрессии.

In [ ]:
rng_demo = np.random.default_rng(SEED)
churned_rows = rng_demo.choice(np.flatnonzero(y_test.to_numpy() == 1), 10, replace=False)
stayed_rows = rng_demo.choice(np.flatnonzero(y_test.to_numpy() == 0), 10, replace=False)
demo_scores = proba_logistic[np.concatenate([churned_rows, stayed_rows])]
demo_labels = np.array([1] * 10 + [0] * 10)
demo_jitter = rng_demo.uniform(-0.22, 0.22, len(demo_labels))
print("оценки ушедших:   ", np.sort(demo_scores[demo_labels == 1]).round(2))
print("оценки оставшихся:", np.sort(demo_scores[demo_labels == 0]).round(2))

Каждый столбец картинки — один порог, и в каждом столбце одни и те же
двадцать клиентов. По вертикали — оценка модели, по горизонтали — небольшой
разброс, чтобы точки не слипались. Кружки — ушедшие, квадраты — оставшиеся,
цвет — клетка матрицы ошибок при этом пороге. Черная черта — порог: всех,
кто выше, модель называет ушедшими. Над столбцом подписаны $TP$ и $FP$,
то есть те, кто выше черты, под столбцом — $FN$ и $TN$, те, кто ниже.

In [ ]:
CELL_COLORS = {"TP": BLUE, "FN": "#E41A1C", "FP": OCHRE, "TN": GREY}


def confusion_cells(labels: np.ndarray, scores: np.ndarray, t: float) -> np.ndarray:
    predicted = scores >= t
    cells = np.where(labels == 1, np.where(predicted, "TP", "FN"), np.where(predicted, "FP", "TN"))
    return cells


demo_thresholds = [0.9, 0.7, 0.5, 0.3, 0.1]
fig, ax = plt.subplots(figsize=(13, 6))
previous = None
tick_labels = []
for col, t in enumerate(demo_thresholds):
    cells = confusion_cells(demo_labels, demo_scores, t)
    moved = np.zeros(len(cells), dtype=bool) if previous is None else cells != previous
    x = col + demo_jitter
    for label, marker in [(1, "o"), (0, "s")]:
        mask = demo_labels == label
        colors = [CELL_COLORS[c] for c in cells[mask]]
        edges = np.where(moved[mask], BLACK, "white")
        ax.scatter(x[mask], demo_scores[mask], c=colors, marker=marker, s=90, edgecolors=edges, linewidths=1.8, zorder=3)
    ax.hlines(t, col - 0.4, col + 0.4, color=BLACK, lw=2.5, zorder=2)
    counts = {name: int((cells == name).sum()) for name in CELL_COLORS}
    ax.text(col, 1.08, f"TP = {counts['TP']}   FP = {counts['FP']}", ha="center", fontsize=10)
    ax.text(col, -0.1, f"FN = {counts['FN']}   TN = {counts['TN']}", ha="center", fontsize=10)
    tick_labels.append(f"t = {t}\nTPR = {counts['TP']}/10, FPR = {counts['FP']}/10")
    previous = cells

handles = [
    Patch(color=CELL_COLORS["TP"], label="TP: ушел, назван ушедшим"),
    Patch(color=CELL_COLORS["FN"], label="FN: ушел, назван оставшимся"),
    Patch(color=CELL_COLORS["FP"], label="FP: остался, назван ушедшим"),
    Patch(color=CELL_COLORS["TN"], label="TN: остался, назван оставшимся"),
    ax.scatter([], [], marker="o", color="white", edgecolors=GREY, s=70, label="на самом деле ушел"),
    ax.scatter([], [], marker="s", color="white", edgecolors=GREY, s=70, label="на самом деле остался"),
    ax.scatter([], [], marker="o", color="white", edgecolors=BLACK, linewidths=1.8, s=70, label="перешел в другую клетку"),
    plt.Line2D([], [], color=BLACK, lw=2.5, label="порог t"),
]
ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=9)
ax.set_xticks(range(len(demo_thresholds)), tick_labels)
ax.set_xlim(-0.6, len(demo_thresholds) - 0.4)
ax.set_ylim(-0.16, 1.14)
ax.set_yticks(np.linspace(0, 1, 6))
ax.set_ylabel("оценка модели: вероятность ухода")
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

Слева направо порог опускается. Объекты, которые черта прошла с прошлого
столбца, обведены черным. Кружки переходят только
из красного $FN$ в синий $TP$, квадраты — только из серого $TN$
в охристый $FP$, и обратно никто не возвращается. В каждом столбце
$TP + FN = 10$ и $FP + TN = 10$.

Так при любом пороге: знаменатели TPR и FPR не меняются, $TP(t) + FN(t) = n_+$ —
все ушедшие, $FP(t) + TN(t) = n_-$ — все оставшиеся, как бы модель их ни назвала.
У precision знаменатель $TP(t) + FP(t)$ с порогом меняется: на картинке
это $0, 2, 7, 10, 13$. Поэтому

$$\mathrm{TPR}(t) = \frac{TP(t)}{n_+}, \qquad \mathrm{FPR}(t) = \frac{FP(t)}{n_-},$$

и с уменьшением порога обе доли не убывают: каждая либо растет, либо остается
прежней. На картинке TPR идет 0, 2, 6, 7, 10 из десяти, FPR — 0, 0, 1, 3, 3
из десяти.

TPR — это recall, доля найденных ушедших. FPR — доля оставшихся, которых
ошибочно назвали ушедшими. Посмотрим на два крайних порога.

- **$t = 1$.** Ушедшим называется объект с оценкой не меньше единицы,
  а вероятность у логистической регрессии всегда меньше единицы. Ушедшим
  не назван никто: $TP(1) = 0$ и $FP(1) = 0$, поэтому
  $\mathrm{TPR}(1) = \frac{0}{n_+} = 0$ и $\mathrm{FPR}(1) = \frac{0}{n_-} = 0$.
  Это точка $(0, 0)$, начало ROC-кривой.
- **$t = 0$.** Любая оценка не меньше нуля, поэтому ушедшими названы все:
  $TP(0) = n_+$ и $FP(0) = n_-$, откуда
  $\mathrm{TPR}(0) = \frac{n_+}{n_+} = 1$ и $\mathrm{FPR}(0) = \frac{n_-}{n_-} = 1$.
  Это точка $(1, 1)$, конец ROC-кривой.

**ROC-кривая** — TPR по вертикали от FPR по горизонтали при всех порогах.
Нарисуем ее для тех же двадцати клиентов. Опускаем порог от 1 до 0,
и каждый раз, когда черта проходит очередного клиента, ставим точку
$(\mathrm{FPR}, \mathrm{TPR})$. Пять порогов с картинки выше отмечены
на кривой.

In [ ]:
demo_fpr, demo_tpr, _ = roc_curve(demo_labels, demo_scores, drop_intermediate=False)
n_pos, n_neg = int(demo_labels.sum()), int((1 - demo_labels).sum())
right_steps = np.flatnonzero(np.diff(demo_fpr) > 0)
bar_left = demo_fpr[right_steps]
bar_cells = np.round(demo_tpr[right_steps] * n_pos).astype(int)

fig, ax = plt.subplots(figsize=(7, 7))
bars = ax.bar(bar_left, bar_cells / n_pos, width=1 / n_neg, align="edge", color="#A9CCEF", edgecolor="white", label="столбики под кривой")
ax.bar_label(bars, labels=[str(k) for k in bar_cells], label_type="center", fontsize=10)
ax.plot(demo_fpr, demo_tpr, color=BLUE, lw=2.5, label="ROC-кривая")
for t in demo_thresholds:
    cells = confusion_cells(demo_labels, demo_scores, t)
    point = ((cells == "FP").sum() / n_neg, (cells == "TP").sum() / n_pos)
    ax.scatter(*point, color=BLACK, s=40, zorder=3)
    ax.annotate(f"t = {t}", point, xytext=(6, 5), textcoords="offset points", fontsize=9)
ax.plot([0, 1], [0, 1], color=BLACK, ls=":", lw=1, label="случайные оценки")
ax.set_xticks(np.linspace(0, 1, n_neg + 1))
ax.set_yticks(np.linspace(0, 1, n_pos + 1))
ax.grid(alpha=0.6)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.set_aspect("equal")
ax.set_xlabel(f"FPR: доля оставшихся, названных ушедшими, шаг 1/{n_neg}")
ax.set_ylabel(f"TPR: доля найденных ушедших, шаг 1/{n_pos}")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

**Единицы на осях.** Обе оси — доли, от 0 до 1, так что кривая живет
в единичном квадрате площади 1. Но шагает она по-разному. Каждый оставшийся,
которого прошел порог, сдвигает ее вправо на $\frac{1}{n_-} = \frac{1}{10}$,
каждый ушедший поднимает вверх на $\frac{1}{n_+} = \frac{1}{10}$. Поэтому сетка
на картинке — $n_- \times n_+ = 10 \times 10$. Одна клетка сетки — прямоугольник
$\frac{1}{n_-} \times \frac{1}{n_+}$ площади $\frac{1}{n_+ n_-} = \frac{1}{100}$,
и она отвечает одной паре «ушедший, оставшийся»: всего таких пар
$n_+ n_- = 100$, столько же и клеток.

**Площадь под кривой** считаем столбиками. Каждый столбик стоит над одним
оставшимся: ширина $\frac{1}{n_-}$, высота — TPR в тот момент, когда порог
его прошел, то есть $\frac{k}{n_+}$, где $k$ — сколько ушедших к этому моменту
уже названы ушедшими. Площадь столбика

$$\frac{1}{n_-} \cdot \frac{k}{n_+} = k \cdot \frac{1}{n_+ n_-},$$

то есть ровно $k$ клеток. На столбиках подписаны эти $k$. Сложим их
и умножим на площадь клетки:

In [ ]:
area_cells = bar_cells.sum()
area = area_cells / (n_pos * n_neg)
print("клеток под кривой по столбикам:", " + ".join(map(str, bar_cells)), "=", area_cells)
print(f"площадь: {area_cells} / {n_pos * n_neg} = {area:.2f}")
print(f"roc_auc_score:  {roc_auc_score(demo_labels, demo_scores):.2f}")

Эта площадь и есть **ROC-AUC**, площадь под ROC-кривой. У модели
со случайными оценками кривая идет вдоль диагонали, площадь 0.5; идеальная
модель сразу поднимается в TPR = 1 при FPR = 0, площадь 1. В общем виде
площадь под кривой TPR от FPR — интеграл:

$$\mathrm{ROC\text{-}AUC} = \int_0^1 \mathrm{TPR}\; d\,\mathrm{FPR}.$$

У ROC-AUC есть смысл без всяких порогов: это вероятность того, что
случайному ушедшему клиенту модель дала оценку выше, чем случайному
оставшемуся. Выведем это. Пусть $n_+$ положительных и $n_-$ отрицательных
объектов, у объекта $i$ ответ $y_i$ (1 — ушел, 0 — остался) и оценка
модели $s_i$, и все $s_i$ разные. Отсортируем объекты по убыванию оценки
и будем опускать порог сверху вниз, пропуская по одному объекту.

- Прошли положительный объект: $TP$ выросло на 1, TPR — на $\frac{1}{n_+}$,
  FPR не изменился. Кривая шагает вверх.
- Прошли отрицательный объект $j$: FPR вырос на $\frac{1}{n_-}$, кривая шагает
  вправо. Высота кривой на этом шаге — текущий TPR, то есть доля
  положительных, которых порог уже пропустил, а это ровно положительные
  с оценкой выше $s_j$.

Площадь под кривой складывается только из шагов вправо — это столбики
с картинки. Шаг на отрицательном $j$ дает прямоугольник ширины $\frac{1}{n_-}$
и высоты $\frac{1}{n_+}\,\#\{i:\ y_i = 1,\ s_i > s_j\}$. На картинке у первого
столбика $k = 6$: у первого по оценке оставшегося, 0.58, выше него шесть
ушедших из десяти.

$$\mathrm{ROC\text{-}AUC} = \sum_{j:\ y_j = 0} \frac{1}{n_-} \cdot \frac{\#\{i:\ y_i = 1,\ s_i > s_j\}}{n_+} = \frac{1}{n_+ n_-} \sum_{j:\ y_j = 0} \#\{i:\ y_i = 1,\ s_i > s_j\}.$$

Сумма в числителе — число пар «положительный, отрицательный», где
положительный выше. Всего пар «положительный, отрицательный» $n_+ n_-$:
каждый из $n_+$ положительных в паре с каждым из $n_-$ отрицательных. Значит, ROC-AUC — доля
правильно упорядоченных пар, то есть вероятность правильного порядка
для случайной пары. Если оценки равны, порог пропускает оба объекта сразу,
кривая идет по диагонали, и такая пара дает половину. В лабораторной
из этой формулы выводится быстрый способ счета через ранги.

**Кривая precision-recall** — precision от recall при всех порогах.
Посмотрим на нее на тех же двадцати клиентах. Отсортируем их по убыванию
оценки и будем опускать порог, пропуская по одному. Когда порог пропустил
первых $k$ клиентов, из которых $TP_k$ ушли, precision и recall равны

$$P_k = \frac{TP_k}{k}, \qquad R_k = \frac{TP_k}{n_+}.$$

На картинке точка — очередной шаг, рядом подписана precision дробью
$\frac{TP_k}{k}$. Цвет отрезка показывает, кем оказался клиент, которого
порог прошел на этом шаге.

In [ ]:
order = np.argsort(-demo_scores)
passed_labels = demo_labels[order]
steps = np.arange(1, len(order) + 1)
tp_passed = np.cumsum(passed_labels)
precision_steps = tp_passed / steps
recall_steps = tp_passed / n_pos
step_colors = [BLUE if label == 1 else "#E41A1C" for label in passed_labels]

fig, ax = plt.subplots(figsize=(8, 5.5))
for j in range(1, len(steps)):
    ax.plot(recall_steps[j - 1:j + 1], precision_steps[j - 1:j + 1], color=step_colors[j], lw=2)
ax.scatter(recall_steps, precision_steps, c=step_colors, s=35, zorder=3)
for j in range(5, len(steps)):
    ax.annotate(f"{tp_passed[j]}/{steps[j]}", (recall_steps[j], precision_steps[j]), xytext=(6, -3), textcoords="offset points", fontsize=8)
ax.axhline(demo_labels.mean(), color=GREY, ls="--", lw=1)
handles = [
    plt.Line2D([], [], color=BLUE, lw=2, label="порог прошел ушедшего: вправо и вверх"),
    plt.Line2D([], [], color="#E41A1C", lw=2, label="порог прошел оставшегося: вниз"),
    plt.Line2D([], [], color=GREY, ls="--", lw=1, label="доля ушедших среди двадцати: 0.5"),
]
ax.legend(handles=handles, loc="lower left", fontsize=9)
ax.set_xlim(0, 1.1)
ax.set_ylim(0.4, 1.05)
ax.set_xticks(np.linspace(0, 1, n_pos + 1))
ax.set_xlabel(f"recall: доля найденных ушедших, шаг 1/{n_pos}")
ax.set_ylabel("precision")
ax.set_title("Кривая precision-recall на двадцати клиентах")
plt.tight_layout()
plt.show()

Первые шесть клиентов по оценке ушли, и precision держится на единице.
Дальше двое оставшихся с оценками 0.58 и 0.48 роняют ее до $\frac{6}{7}$
и $\frac{6}{8}$, ушедший с 0.41 поднимает до $\frac{7}{9}$, оставшийся
с 0.37 роняет до $\frac{7}{10}$, три ушедших подряд поднимают до $\frac{10}{13}$,
и семь оставшихся в конце опускают ее до $\frac{10}{20} = 0.5$ — доли ушедших
среди всех двадцати.

**Почему кривая то растет, то падает.** Посмотрим, что происходит
с $P_k = \frac{TP_k}{k}$, когда порог проходит еще одного клиента.

- **Клиент ушел.** Растут и $TP$, и $k$: $P_{k+1} = \frac{TP_k + 1}{k + 1}$,
  recall растет на $\frac{1}{n_+}$. Сравним с прошлым шагом:

  $$\frac{TP_k + 1}{k + 1} \ge \frac{TP_k}{k} \iff k\,(TP_k + 1) \ge TP_k\,(k + 1) \iff k \ge TP_k.$$

  Это верно всегда: ушедших среди пропущенных не больше, чем всех
  пропущенных. Precision не падает, кривая идет вправо и вверх.
- **Клиент остался.** $TP$ прежний, $k$ растет:
  $P_{k+1} = \frac{TP_k}{k + 1} < \frac{TP_k}{k}$ при $TP_k > 0$, а recall
  не меняется. Кривая идет строго вниз.

Отсюда зубцы: каждый оставшийся клиент с высокой оценкой роняет precision,
каждый следующий ушедший немного ее поднимает. У ROC-кривой так не бывает:
при опускании порога TPR и FPR не убывают, и она идет только вправо и вверх.

Площадь под кривой precision-recall считают как **average precision**:

$$\mathrm{AP} = \sum_k (R_k - R_{k-1})\, P_k.$$

Пусть у всех объектов разные оценки. Recall растет только на тех
шагах, где очередной объект положительный, и каждый раз на $\frac{1}{n_+}$, где $n_+$ — число положительных. На остальных
шагах $R_k - R_{k-1} = 0$. Значит, в сумме остаются только положительные
объекты:

$$\mathrm{AP} = \frac{1}{n_+} \sum_{i:\ y_i = 1} P_{k(i)},$$

где $k(i)$ — место объекта $i$ в сортировке. Average precision — средняя
precision, которую видит каждый положительный объект, если поставить порог
ровно на него. У случайной модели precision на любом пороге примерно равна
доле положительных, и AP тоже.

Проверим на двадцати клиентах: возьмем precision в те моменты, когда порог
проходит каждого из десяти ушедших, и усредним.

In [ ]:
at_positives = passed_labels == 1
fractions = [f"{tp}/{k}" for tp, k in zip(tp_passed[at_positives], steps[at_positives])]
print("precision у каждого ушедшего:", ", ".join(fractions))
print(f"среднее, то есть AP руками: {precision_steps[at_positives].mean():.3f}")
print(f"average_precision_score:     {average_precision_score(demo_labels, demo_scores):.3f}")

### Все модели в одном сетапе

Одна функция обучает модель, считает одни и те же метрики на одном тестовом сете
и добавляет строку в таблицу `results`.

In [ ]:
results = {}
scores_on_test = {}


def evaluate(name, model):
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    predicted = (proba >= 0.5).astype(int)
    scores_on_test[name] = proba
    results[name] = {
        "accuracy": accuracy_score(y_test, predicted),
        "precision": precision_score(y_test, predicted, zero_division=0),
        "recall": recall_score(y_test, predicted),
        "F1": f1_score(y_test, predicted),
        "ROC-AUC": roc_auc_score(y_test, proba),
        "PR-AUC": average_precision_score(y_test, proba),
    }
    table = pd.DataFrame(results).T
    return table

Три модели: константа, которая всегда отвечает «остался», логистическая
регрессия и она же с весами классов.

In [ ]:
dummy_model = make_model(DummyClassifier(strategy="most_frequent"))
logistic_plain = make_model(LogisticRegression(max_iter=2000))
logistic_balanced = make_model(LogisticRegression(max_iter=2000, class_weight="balanced"))

evaluate("всегда «остался»", dummy_model)
evaluate("логистическая регрессия", logistic_plain)
metrics_table = evaluate("логистическая, class_weight=balanced", logistic_balanced)
show_table(metrics_table)

In [ ]:
plot_metric_bars(metrics_table, "Три модели на одном тестовом сете")

Модель, которая всегда отвечает «остался», набирает accuracy 0.735 и не находит
ни одного ушедшего. Accuracy при дисбалансе почти бесполезна: разница между
ничего не умеющей моделью и хорошей — семь процентных пунктов. Precision
и recall показывают, что на самом деле происходит.

`class_weight="balanced"` поднимает recall ценой precision. Разберемся,
почему это почти то же самое, что сдвинуть порог.

**Что делают веса классов.** Без весов каждый клиент обучения входит
в функцию потерь с весом 1. С `class_weight="balanced"` объект класса $k$
входит с весом

$$v_k = \frac{n}{K \cdot n_k},$$

где $n$ — число объектов обучения, $K = 2$ — число классов, $n_k$ — сколько
объектов класса $k$. Если $n_k = \frac{n}{K}$, то есть классы поровну, вес
равен 1. Редкий класс получает вес больше единицы, частый — меньше:

In [ ]:
class_share = y_train.value_counts(normalize=True).sort_index()
balanced_weights = 1 / (2 * class_share)
weights_table = pd.DataFrame({"доля в обучении": class_share, "вес v_k": balanced_weights}).rename(index={0: "остался", 1: "ушел"}).rename_axis(None)
show_table(weights_table, digits=2)

Ошибка на ушедшем стоит почти втрое дороже ошибки на оставшемся,
$\frac{1.88}{0.68} \approx 2.8$. Модели выгоднее чаще говорить «ушел», и она
поднимает оценки всем клиентам.

**Логит.** Вспомним третье занятие. Логистическая регрессия считает
для клиента $x$ число

$$z = w^\top x + b,$$

логит, и превращает его в вероятность $p = \sigma(z) = \frac{1}{1 + e^{-z}}$.
Порог 0.5 по вероятности — это порог 0 по логиту: $\sigma(0) = \frac{1}{1 + 1} = 0.5$,
а сигмоида возрастает, поэтому $\sigma(z) \ge 0.5 \iff z \ge 0$. Метод
`decision_function` возвращает как раз $z$. Сравним логиты двух моделей
на каждом клиенте тестового сета: по горизонтали логит без весов, по вертикали —
с весами.

In [ ]:
logit_plain = logistic_plain.decision_function(X_test)
logit_balanced = logistic_balanced.decision_function(X_test)
logit_shift = logit_balanced - logit_plain
mean_shift = logit_shift.mean()

line = np.array([logit_plain.min(), logit_plain.max()])
plt.figure(figsize=(6.5, 6))
plt.scatter(logit_plain, logit_balanced, s=6, alpha=0.4, color=BLUE, label="клиенты тестового сета")
plt.plot(line, line, color=GREY, ls="--", lw=1.5, label="логит не изменился")
plt.plot(line, line + mean_shift, color=OCHRE, lw=2, label=f"логит вырос на {mean_shift:.2f}")
plt.axhline(0, color=BLACK, lw=0.8)
plt.axvline(0, color=BLACK, lw=0.8)
plt.axvline(-mean_shift, color=OCHRE, ls=":", lw=2, label=f"z = {-mean_shift:.2f}: здесь логит с весами равен 0")
plt.xlabel("логит без весов классов")
plt.ylabel("логит с весами классов")
plt.legend()
plt.show()
print(f"сдвиг логита по клиентам: среднее {mean_shift:.2f}, разброс {logit_shift.std():.2f}")

Точки легли вдоль прямой, параллельной диагонали: у всех клиентов логит
вырос почти на одно и то же число $c \approx 1.04$, разброс всего 0.07.
А сдвиг всех логитов на $c$ — это сдвиг порога. Модель с весами называет
клиента ушедшим, когда ее логит $z + c$ не меньше нуля:

$$z + c \ge 0 \iff z \ge -c \iff \sigma(z) \ge \sigma(-c).$$

На картинке это пунктирная вертикаль: все точки правее нее модель с весами
называет ушедшими. Последний шаг — снова потому, что сигмоида возрастает. Справа стоит
вероятность модели без весов. Значит, модель с весами при пороге 0.5
почти то же самое, что модель без весов при пороге

$$t = \sigma(-c) = \frac{1}{1 + e^{c}} = \frac{1}{1 + e^{1.04}} \approx \frac{1}{1 + 2.83} \approx 0.26.$$

Проверим на тестовом сете:

In [ ]:
equivalent_threshold = 1 / (1 + np.exp(mean_shift))
proba_plain = logistic_plain.predict_proba(X_test)[:, 1]
proba_balanced = logistic_balanced.predict_proba(X_test)[:, 1]
labels_balanced = (proba_balanced >= 0.5).astype(int)
labels_shifted = (proba_plain >= equivalent_threshold).astype(int)
threshold_table = pd.DataFrame({
    "с весами, порог 0.5": [precision_score(y_test, labels_balanced), recall_score(y_test, labels_balanced), f1_score(y_test, labels_balanced)],
    f"без весов, порог {equivalent_threshold:.2f}": [precision_score(y_test, labels_shifted), recall_score(y_test, labels_shifted), f1_score(y_test, labels_shifted)],
}, index=["precision", "recall", "F1"])
show_table(threshold_table, axis=None)

Метрики почти совпадают: веса классов в логистической регрессии почти
не меняют саму модель, а сдвигают все логиты на одно число, то есть порог.
Сам свободный член $b$ при этом меняется меньше, чем на $c$: у столбцов
one-hot одного признака в каждой строке ровно одна единица, поэтому прибавка
ко всем весам категорий признака действует так же, как прибавка к $b$,
и модель раскладывает сдвиг между ними. Как метрики по меткам зависят
от порога:

In [ ]:
thresholds = np.round(np.arange(0.05, 0.96, 0.05), 2)
curves = {"precision": [], "recall": [], "F1": []}
for t in thresholds:
    labels = (proba_logistic >= t).astype(int)
    curves["precision"].append(precision_score(y_test, labels, zero_division=np.nan))
    curves["recall"].append(recall_score(y_test, labels))
    curves["F1"].append(f1_score(y_test, labels))

plt.figure(figsize=(8, 3.8))
for (name, values), color in zip(curves.items(), [BLUE, OCHRE, BLACK]):
    plt.plot(thresholds, values, "o-", ms=3, color=color, label=name)
plt.axvline(0.5, color=GREY, ls="--", lw=1)
plt.xlabel("порог")
plt.title("Логистическая регрессия: метрики по меткам при разных порогах")
plt.ylim(0, 1.05)
plt.legend()
plt.show()

Проверим смысл ROC-AUC как вероятности правильного порядка пары, перебрав
все пары «ушедший — оставшийся» в тестовом сете.

In [ ]:
positive = proba_logistic[y_test.to_numpy() == 1]
negative = proba_logistic[y_test.to_numpy() == 0]
pairs_won = (positive[:, None] > negative[None, :]).mean() + 0.5 * (positive[:, None] == negative[None, :]).mean()
print(f"доля пар, где ушедший выше оставшегося: {pairs_won:.4f}")
print(f"roc_auc_score:                           {roc_auc_score(y_test, proba_logistic):.4f}")

Картинка к этому числу: распределения оценок модели у ушедших и оставшихся.
Чем меньше они перекрываются, тем чаще случайный ушедший оказывается выше
случайного оставшегося.

In [ ]:
plt.figure(figsize=(8, 3.6))
for value, color, label in [(0, GREY, "остался"), (1, OCHRE, "ушел")]:
    plt.hist(proba_logistic[y_test.to_numpy() == value], bins=30, range=(0, 1), alpha=0.6, density=True, color=color, label=label)
plt.xlabel("вероятность ухода по модели")
plt.title("Оценки логистической регрессии на тестовом сете")
plt.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for (name, proba), color in zip(scores_on_test.items(), MODEL_COLORS):
    fpr, tpr, _ = roc_curve(y_test, proba)
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f"{name}: {roc_auc_score(y_test, proba):.3f}")
    if name == "всегда «остался»":
        continue
    precision_curve, recall_curve, _ = precision_recall_curve(y_test, proba)
    axes[1].plot(recall_curve, precision_curve, color=color, lw=2, drawstyle="steps-post",
                 label=f"{name}: {average_precision_score(y_test, proba):.3f}")
axes[0].plot([0, 1], [0, 1], color=BLACK, lw=1, ls=":")
axes[0].set_xlabel("FPR: доля оставшихся, названных ушедшими")
axes[0].set_ylabel("TPR: доля найденных ушедших")
axes[0].set_title("ROC-кривая")
axes[1].axhline(y_test.mean(), color=GREY, lw=2, ls=":", label=f"всегда «остался»: {y_test.mean():.3f}, случайная модель около этого уровня")
axes[1].set_xlabel("recall")
axes[1].set_ylabel("precision")
axes[1].set_title("Кривая precision-recall")
for ax in axes:
    ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

Что видно на двух графиках.

- **Случайная модель.** На ROC-кривой это диагональ, площадь под ней 0.5.
  На кривой precision-recall — горизонталь на уровне доли положительного
  класса, здесь 0.265: какой порог ни поставь, среди названных ушедшими
  ушедших будет примерно столько же, сколько во всем тестовом сете. Константа
  «всегда остался» дает ровно эти значения: ROC-AUC 0.500 и PR-AUC 0.265.
- **Почему PR-AUC честнее при сильном дисбалансе.** В FPR ложные тревоги
  делятся на всех оставшихся: $\mathrm{FPR} = \frac{FP}{n_-}$. В precision
  те же ложные тревоги стоят рядом с верно найденными: $P = \frac{TP}{TP + FP}$.
  Пусть оставшихся миллион, ушедших тысяча, и модель нашла всех ушедших ценой
  ста тысяч ложных тревог. Тогда $\mathrm{TPR} = \frac{1000}{1000} = 1$
  и $\mathrm{FPR} = \frac{100\,000}{1\,000\,000} = 0.1$: точка на ROC-кривой
  выглядит отлично. Но $P = \frac{1000}{1000 + 100\,000} \approx 0.01$:
  на одного настоящего ушедшего сотня ошибочных, и precision это сразу видит.
- **Веса классов почти не меняют кривые.** Обе кривые зависят только от того,
  в каком порядке модель расставила объекты по оценке. `class_weight="balanced"`
  сдвигает логит у всех объектов почти на одно и то же число, а сигмоида
  монотонна, поэтому порядок по вероятности почти не меняется, хотя сами
  вероятности сдвигаются по-разному. Поэтому ROC-AUC 0.847 и 0.846,
  PR-AUC 0.638 и 0.636. Меняются метрики по меткам при пороге 0.5: recall
  вырос с 0.557 до 0.797, precision упала с 0.663 до 0.519.
- **Ступеньки на кривой precision-recall.** На шаге $k$ recall прирастает
  на $R_k - R_{k-1}$, а precision равна $P_k$, так что каждая ступенька —
  прямоугольник шириной $R_k - R_{k-1}$ и высотой $P_k$. Площадь под
  ступеньками — сумма этих прямоугольников, $\sum_k (R_k - R_{k-1})\, P_k$,
  то есть в точности average precision из формулы выше.

### Несколько классов: усреднение

В задаче с $K$ классами для каждого класса $k$ считают свои $TP_k$, $FP_k$,
$FN_k$, как в задаче «класс $k$ против остальных», и из них свою $F_1^{(k)}$.
Дальше их усредняют.

- **macro**: простое среднее, каждый класс весит одинаково, как бы редок он ни был:

$$F_1^{\mathrm{macro}} = \frac{1}{K}\sum_{k=1}^{K} F_1^{(k)}.$$

- **weighted**: среднее с весами $\frac{n_k}{n}$, где $n_k$ — число объектов класса $k$:

$$F_1^{\mathrm{weighted}} = \sum_{k=1}^{K} \frac{n_k}{n}\, F_1^{(k)}.$$

- **micro**: сначала складываем клетки всех классов, потом считаем метрику:

$$P^{\mathrm{micro}} = \frac{\sum_k TP_k}{\sum_k (TP_k + FP_k)}, \qquad R^{\mathrm{micro}} = \frac{\sum_k TP_k}{\sum_k (TP_k + FN_k)}.$$

Когда у каждого объекта ровно одна метка, micro совпадает с accuracy.
Каждая ошибка «класс $a$ назван классом $b$» — это ровно один $FP$ у класса
$b$ и ровно один $FN$ у класса $a$. Поэтому $\sum_k FP_k = \sum_k FN_k$ =
число ошибок, а $\sum_k TP_k$ = число верных ответов. Знаменатели обеих
дробей равны $n$, и $P^{\mathrm{micro}} = R^{\mathrm{micro}} = F_1^{\mathrm{micro}} = \mathrm{accuracy}$.
При дисбалансе macro показывает, как модель справляется с редкими
классами, а micro и accuracy этого почти не видят.

## 8. Кросс-валидация

Одно разбиение на обучение и тестовый сет — одна случайная выборка. Насколько
сильно метрика зависит от того, как именно разбили?

In [ ]:
split_scores = []
for seed in range(20):
    a_train, a_test, b_train, b_test = train_test_split(feature_frame, target, test_size=0.25, random_state=seed, stratify=target)
    split_model = make_model(LogisticRegression(max_iter=2000))
    split_model.fit(a_train, b_train)
    split_scores.append(roc_auc_score(b_test, split_model.predict_proba(a_test)[:, 1]))

plt.figure(figsize=(8, 3.5))
plt.hist(split_scores, bins=12, color=BLUE)
plt.gca().yaxis.set_major_locator(MaxNLocator(integer=True))
plt.xlabel("ROC-AUC на тестовом сете")
plt.ylabel("число разбиений")
plt.title("Одна и та же модель, 20 разных разбиений")
plt.show()
print(f"от {min(split_scores):.3f} до {max(split_scores):.3f}")

Разница между худшим и лучшим разбиением — три сотых. Значит, одно число
с одного разбиения шумит на три сотых: его нельзя сравнивать с числом,
полученным на другом разбиении или в другой работе.

Кросс-валидация делит данные на $K$ фолдов, $K$ раз обучает модель
на $K - 1$ фолдах, оценивает на оставшемся и усредняет. Фолд, на котором
модель оценивают в очередном разбиении, называют **валидационной частью**.
Каждый объект ровно один раз попадает в валидационную часть.

**Валидационная часть и тестовый сет — не одно и то же.** Тестовый сет
откладывают один раз, в самом начале, как `X_test` и `y_test` в разделе 7,
и дальше не трогают до финальной оценки.

| | валидационная часть | тестовый сет |
|---|---|---|
| откуда берется | фолд обучающих данных, в каждом разбиении свой | отложен один раз, до всякого обучения |
| сколько раз используется | $K$ раз, каждый объект обучения ровно один раз | один раз, в самом конце |
| для чего | выбрать модель, гиперпараметры, признаки, порог | оценить уже выбранную модель |
| насколько честна оценка | после выбора оптимистична: из многих вариантов взяли лучший | честная, если по тестовому сету ничего не выбирали |

Главное правило: по тестовому сету ничего не выбирают. Если по нему
сравнили две модели и оставили лучшую, он стал валидационной частью,
и честной оценки больше нет. В разделах 7 и 10 мы сравниваем модели прямо
на тестовом сете, чтобы показать метрики; в работе такое сравнение делают
кросс-валидацией на обучающих данных. Ниже для простоты кросс-валидация
идет по всем клиентам, а в работе ее запускают только на обучении,
не трогая тестовый сет.

### Схемы разбиения

Нарисуем, какие объекты попадают в валидационную часть в каждой схеме, на игрушечных
данных из 20 объектов. Классов три, 10, 5 и 5 объектов, и данные
отсортированы по классу, как часто бывает в выгрузках. Каждая строка —
одно разбиение: серые квадраты с желтым контуром идут в обучение, цветные — в валидационную часть,
и цвет — это класс объекта. Нижняя строка — классы всех объектов.

In [ ]:
toy_n = 20
toy_X = np.zeros((toy_n, 1))
toy_y = np.array([0] * 10 + [1] * 5 + [2] * 5)
toy_groups = np.arange(toy_n) % 5
print("классы: ", toy_y)
print("группы: ", toy_groups)

Функции, которые рисуют схемы: квадрат, оформление осей и схема целиком.

In [ ]:
TRAIN_COLOR = "#E5E7EB"
TRAIN_EDGE = "#F2B705"
GROUP_COLORS = ["#984EA3", "#FF7F00", "#A65628", "#F781BF", "#FFD92F"]


def draw_square(ax, x, y, color, edge="none", gap=0.12):
    square = Rectangle((x + gap / 2, y + gap / 2), 1 - gap, 1 - gap, facecolor=color, edgecolor=edge, lw=1.5)
    ax.add_patch(square)


def style_scheme_axes(ax, labels, title):
    ax.set_xlim(0, toy_n)
    ax.set_ylim(len(labels), 0)
    ax.set_aspect("equal")
    ax.set_yticks(np.arange(len(labels)) + 0.5, labels, fontsize=8)
    ax.set_xticks(np.arange(toy_n) + 0.5, range(toy_n), fontsize=7)
    ax.tick_params(length=0)
    ax.grid(False)
    for side in ax.spines.values():
        side.set_visible(False)
    ax.set_title(title, fontsize=10, loc="left")


def draw_schemes(schemes):
    folds_by_scheme = {}
    for name, (splitter, groups) in schemes.items():
        split_groups = groups if isinstance(splitter, GroupKFold) else None
        folds_by_scheme[name] = list(splitter.split(toy_X, toy_y, groups=split_groups))
    heights = [len(folds) + 1 + (groups is not None) for folds, (_, groups) in zip(folds_by_scheme.values(), schemes.values())]
    fig, axes = plt.subplots(len(schemes), 1, figsize=(8.5, 0.36 * sum(heights) + 0.55 * len(schemes)),
                             gridspec_kw={"height_ratios": heights})
    for ax, (name, (splitter, groups)) in zip(axes, schemes.items()):
        folds = folds_by_scheme[name]
        repeats = getattr(splitter, "n_repeats", 1)
        per_repeat = len(folds) // repeats
        labels = []
        for row, (train_idx, test_idx) in enumerate(folds):
            for i in train_idx:
                draw_square(ax, i, row, TRAIN_COLOR, edge=TRAIN_EDGE)
            for i in test_idx:
                draw_square(ax, i, row, CLASS_COLORS[toy_y[i]])
            if repeats > 1:
                labels.append(f"повтор {row // per_repeat + 1}, разбиение {row % per_repeat + 1}")
            else:
                labels.append(f"разбиение {row + 1}")
        for r in range(1, repeats):
            ax.axhline(r * per_repeat, color=BLACK, lw=1.2)
        for i in range(toy_n):
            draw_square(ax, i, len(folds), CLASS_COLORS[toy_y[i]])
        labels.append("класс")
        if groups is not None:
            for i in range(toy_n):
                draw_square(ax, i, len(folds) + 1, GROUP_COLORS[groups[i]])
            labels.append("группа")
        style_scheme_axes(ax, labels, name)
    handles = [Patch(facecolor=TRAIN_COLOR, edgecolor=TRAIN_EDGE, lw=1.5, label="обучение")] + [
        Patch(color=CLASS_COLORS[k], label=f"валидационная часть, класс {k}") for k in range(3)
    ]
    fig.legend(handles=handles, loc="upper center", ncol=4, frameon=False, fontsize=9, bbox_to_anchor=(0.5, 1.02))
    plt.tight_layout()
    plt.show()

In [ ]:
draw_schemes({
    "KFold, без перемешивания": (KFold(5), None),
    "KFold, с перемешиванием": (KFold(5, shuffle=True, random_state=SEED), None),
    "StratifiedKFold": (StratifiedKFold(5, shuffle=True, random_state=SEED), None),
})

**KFold без перемешивания** режет подряд. Данные отсортированы по классу,
поэтому в первых двух валидационных частях только класс 0, а в последней — только
класс 2, и в обучении от класса 2 остается один объект из пяти. Доли классов
в обучении и в валидационной части совсем не такие, как в данных.

**KFold с перемешиванием** лучше, но доли классов в валидационных частях скачут
от разбиения к разбиению: в первой валидационной части нет ни одного объекта класса 1,
во второй и в пятой — ни одного объекта класса 2.

**StratifiedKFold** берет объекты каждого класса отдельно, перемешивает
их и раздает по частям поровну. Класса 0 десять объектов на пять частей —
по два в каждую валидационную часть, классов 1 и 2 по пять — по одному.
В каждой валидационной части ровно та же доля каждого класса, что во всех данных.

Еще три схемы.

- **GroupKFold** — для данных, где у объектов есть группы: несколько
  диалогов одного пользователя, несколько снимков одного пациента. Группа
  целиком попадает либо в обучение, либо в валидационную часть. В игрушечных данных
  пять групп по четыре объекта, объекты одной группы разбросаны по всему
  списку, цвет группы — в нижней строке. Для сравнения первым нарисован
  обычный KFold с перемешиванием на тех же группах: все пять групп
  оказываются разорваны между обучением и валидационной частью.
- **TimeSeriesSplit** всегда берет в валидационную часть объекты, которые
  идут позже обучения, и обучающая часть с каждым разбиением растет.
- **ShuffleSplit** просто несколько раз случайно откладывает часть объектов
  в валидационную часть.

In [ ]:
draw_schemes({
    "KFold с перемешиванием: группы разорваны": (KFold(5, shuffle=True, random_state=SEED), toy_groups),
    "GroupKFold: группа целиком в одной части": (GroupKFold(5), toy_groups),
    "TimeSeriesSplit: валидационная часть всегда позже обучения": (TimeSeriesSplit(4), None),
    "ShuffleSplit: случайные подвыборки": (ShuffleSplit(5, test_size=0.2, random_state=SEED), None),
})

- **GroupKFold** не разрывает группы: иначе модель выучит пользователя,
  а не задачу. В домашней работе с диалогами это важно.
- **TimeSeriesSplit** нужен, когда прогнозируют будущее: обучаться
  на будущем нельзя.
- **ShuffleSplit** — несколько независимых случайных разбиений: валидационные
  части могут пересекаться, а некоторые объекты не попадут в валидационную
  часть ни разу. На картинке объект 0 попал в нее три раза, а объекты 3, 4, 9
  и с 11 по 14 — ни разу.

**Чем ShuffleSplit отличается от KFold.** Обе схемы перемешивают данные,
но режут их по-разному.

| | KFold | ShuffleSplit |
|---|---|---|
| как делит | один раз режет перемешанные данные на $K$ частей | каждое разбиение заново случайно откладывает долю `test_size` |
| валидационные части | не пересекаются и вместе дают все данные | независимы, могут пересекаться |
| сколько раз объект в валидационной части | ровно один раз | сколько выпадет: 0, 1, 2 и больше |
| размер валидационной части | $\frac{n}{K}$, задан числом частей | `test_size`, не зависит от числа разбиений |
| число разбиений | $K$ | `n_splits`, любое |

Посчитаем по картинкам, сколько раз каждый из двадцати объектов попал
в валидационную часть у KFold с перемешиванием и у ShuffleSplit.

In [ ]:
kfold_splits = KFold(5, shuffle=True, random_state=SEED).split(toy_X)
shuffle_splits = ShuffleSplit(5, test_size=0.2, random_state=SEED).split(toy_X)
times_in_check = {
    "KFold, 5 частей": np.bincount(np.concatenate([test for _, test in kfold_splits]), minlength=toy_n),
    "ShuffleSplit, 5 разбиений по 20%": np.bincount(np.concatenate([test for _, test in shuffle_splits]), minlength=toy_n),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 3.2), sharey=True)
for ax, (name, counts), color in zip(axes, times_in_check.items(), [BLUE, OCHRE]):
    ax.bar(np.arange(toy_n), counts, color=color)
    ax.set_xticks(np.arange(toy_n))
    ax.set_xlabel("номер объекта")
    ax.set_title(name)
axes[0].set_ylabel("раз в валидационной части")
axes[0].yaxis.set_major_locator(MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

У KFold каждый объект ровно один раз попадает в валидационную часть, поэтому
по его валидационным частям можно собрать предсказание для каждой строки — так устроен
кросс-фиттинг в target encoding. У ShuffleSplit одни объекты попали
в валидационную часть трижды, другие ни разу, зато размер валидационной
части и число разбиений задаются
независимо. На больших данных это экономит время: три разбиения с валидационной частью
по 10% — это три обучения, а KFold с той же долей валидационной части обучает модель
десять раз.

Еще две схемы.

- **Leave-One-Out** — крайний случай KFold с $K = n$: в каждой валидационной
  части ровно один объект, и разбиений столько же, сколько объектов. Модель учится
  почти на всех данных, но обучений $n$ вместо пяти; на наших семи тысячах
  клиентов это семь тысяч моделей.
- **RepeatedStratifiedKFold** повторяет StratifiedKFold несколько раз
  с разным перемешиванием. Внутри повтора каждый объект ровно один раз
  попадает в валидационную часть, а от повтора к повтору части меняются. Черная линия
  на картинке отделяет повторы.

In [ ]:
draw_schemes({
    "LeaveOneOut: в валидационной части один объект": (LeaveOneOut(), None),
    "RepeatedStratifiedKFold: StratifiedKFold два раза с разным перемешиванием": (
        RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=SEED), None),
})

- **Leave-One-Out.** Валидационная часть идет по диагонали: объект $i$
  попадает в нее в разбиении $i + 1$. Стратифицировать тут нечего,
  в валидационной части один объект.
  Метрики вроде ROC-AUC на одном объекте не посчитать, поэтому их считают
  один раз по всем $n$ предсказаниям сразу.
- **RepeatedStratifiedKFold.** В каждом повторе в каждой валидационной части два объекта
  класса 0 и по одному классов 1 и 2, как у StratifiedKFold, но состав частей
  во втором повторе другой. Зачем это нужно, видно дальше на наших данных.

### Кросс-валидация на наших данных

`cross_validate` считает несколько метрик сразу и с `return_train_score=True`
возвращает их и на обучающих частях.

In [ ]:
cv_churn = StratifiedKFold(5, shuffle=True, random_state=SEED)
cv_model = make_model(LogisticRegression(max_iter=2000))
cv_scores = cross_validate(
    cv_model, feature_frame, target, cv=cv_churn,
    scoring=["roc_auc", "average_precision", "f1"], return_train_score=True,
)
cv_table = pd.DataFrame(cv_scores).drop(columns=["fit_time", "score_time"])
cv_table.index = [f"разбиение {k + 1}" for k in range(len(cv_table))]
show_table(cv_table)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
folds = np.arange(1, len(cv_table) + 1)
for ax, metric in zip(axes, ["roc_auc", "average_precision", "f1"]):
    ax.plot(folds, cv_table[f"train_{metric}"], "o-", color=GREY, label="обучающая часть")
    ax.plot(folds, cv_table[f"test_{metric}"], "o-", color=BLUE, label="валидационная часть")
    ax.set_title(metric)
    ax.set_xlabel("номер разбиения")
    ax.set_xticks(folds)
axes[0].legend(fontsize=9)
plt.tight_layout()
plt.show()

Метрики на обучении и на валидационной части близки: логистическая регрессия на этих
данных не переобучается. Итоговый результат модели — среднее по частям
и разброс, а не одно число. `RepeatedStratifiedKFold` повторяет пятикратное
разбиение четыре раза с разным перемешиванием и дает двадцать оценок вместо
пяти. Разброс отдельных оценок от этого в среднем не меняется: каждая
по-прежнему посчитана на одной пятой данных.

In [ ]:
repeated_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=SEED)
single = cross_val_score(cv_model, feature_frame, target, cv=cv_churn, scoring="roc_auc")
repeated = cross_val_score(cv_model, feature_frame, target, cv=repeated_cv, scoring="roc_auc")

plt.figure(figsize=(8, 2.8))
for y, (name, scores), color in [(1, ("StratifiedKFold, 5 оценок", single), BLUE), (0, ("RepeatedStratifiedKFold, 20 оценок", repeated), OCHRE)]:
    plt.scatter(scores, np.full(len(scores), y) + np.random.default_rng(y).uniform(-0.08, 0.08, len(scores)), color=color, s=25)
    plt.plot([scores.mean()] * 2, [y - 0.25, y + 0.25], color=BLACK, lw=2)
plt.yticks([0, 1], ["RepeatedStratifiedKFold, 20 оценок", "StratifiedKFold, 5 оценок"])
plt.xlabel("ROC-AUC на валидационной части; черная черта — среднее")
plt.ylim(-0.6, 1.6)
plt.show()
print(f"разброс отдельных оценок: 5 оценок {single.std():.3f}, 20 оценок {repeated.std():.3f}")

Разбросы 0.013 и 0.010 отличаются случайно: разброс, посчитанный всего
по пяти числам, сам шумный. Что дают повторы, видно по средним: у каждого
из четырех повторов свое перемешивание и свое среднее.

In [ ]:
repeat_means = repeated.reshape(4, 5).mean(axis=1)
repeat_spreads = repeated.reshape(4, 5).std(axis=1)
print("среднее каждого повтора: ", repeat_means.round(4))
print("разброс внутри повтора:  ", repeat_spreads.round(3))

Средние повторов отличаются друг от друга на тысячные, а отдельные оценки —
на сотые: от того, как перемешали данные, итоговое среднее почти не зависит,
и повторы убирают именно эту случайность. Делить разброс на $\sqrt{20}$,
как для независимых оценок, нельзя: все двадцать посчитаны на одних и тех же
семи тысячах клиентов, и неопределенность от того, что клиентов всего семь
тысяч, повторы не уменьшают.

### Утечка через предобработку

Все, что учится на данных — масштабирование, заполнение пропусков,
кодирование, отбор признаков, — должно учиться только на обучающей части
каждого разбиения. Самый наглядный пример: 200 объектов, 5000 признаков
из чистого шума и случайный ответ. Выберем $k$ признаков, сильнее всего
связанных с ответом, двумя способами:

- **с утечкой**: отбираем по всем данным, потом считаем кросс-валидацию
  на отобранных;
- **честно**: отбор — шаг конвейера, и кросс-валидация делает его заново
  в каждом разбиении только по обучающей части.

Ответ случайный, так что честная accuracy должна быть около 0.5 при любом $k$.

In [ ]:
rng_noise = np.random.default_rng(1)
X_noise = rng_noise.normal(size=(200, 5000))
y_noise = rng_noise.integers(0, 2, 200)
print("признаки:", X_noise.shape, " доля единиц в ответе:", y_noise.mean())

Сначала с утечкой, для 20 признаков. Отбор делается один раз по всем
200 объектам, а кросс-валидация потом идет уже по отобранным столбцам.

In [ ]:
selector = SelectKBest(f_classif, k=20)
selected_on_all = selector.fit_transform(X_noise, y_noise)
leaky_model = LogisticRegression(max_iter=1000)
leaky_cv = cross_val_score(leaky_model, selected_on_all, y_noise, cv=5)
print("с утечкой, accuracy по частям:", leaky_cv.round(2), " среднее:", round(leaky_cv.mean(), 3))

Теперь честно: отбор — первый шаг конвейера, и кросс-валидация заново
отбирает 20 признаков в каждом разбиении только по обучающей части.

In [ ]:
honest_pipeline = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=1000))
honest_cv = cross_val_score(honest_pipeline, X_noise, y_noise, cv=5)
print("честно, accuracy по частям:", honest_cv.round(2), " среднее:", round(honest_cv.mean(), 3))

То же самое для разного числа отобранных признаков.

In [ ]:
k_values = [5, 10, 20, 50, 100, 200]
leaky_scores, honest_scores = [], []
for k in k_values:
    selector = SelectKBest(f_classif, k=k)
    selected_on_all = selector.fit_transform(X_noise, y_noise)
    leaky_model = LogisticRegression(max_iter=1000)
    leaky_cv = cross_val_score(leaky_model, selected_on_all, y_noise, cv=5)
    leaky_scores.append(leaky_cv.mean())

    honest_pipeline = make_pipeline(SelectKBest(f_classif, k=k), LogisticRegression(max_iter=1000))
    honest_cv = cross_val_score(honest_pipeline, X_noise, y_noise, cv=5)
    honest_scores.append(honest_cv.mean())

plt.figure(figsize=(8, 3.8))
plt.plot(k_values, leaky_scores, "o-", color=OCHRE, lw=2, label="отбор по всем данным, с утечкой")
plt.plot(k_values, honest_scores, "o-", color=BLUE, lw=2, label="отбор внутри конвейера, честно")
plt.axhline(0.5, color=BLACK, ls="--", lw=1, label="случайное угадывание")
plt.xscale("log")
plt.xticks(k_values, k_values)
plt.xlabel("сколько признаков отобрано")
plt.ylabel("accuracy на кросс-валидации")
plt.ylim(0.3, 1.02)
plt.title("Признаки — чистый шум, ответ случайный")
plt.legend(fontsize=9)
plt.show()

Из пяти тысяч случайных признаков какие-то случайно совпали с ответом на этих
двухстах объектах. Отобрав их по всем данным, мы подсмотрели ответ
валидационных частей, и кросс-валидация показывает accuracy намного выше 0.5
на чистом шуме. Чем больше таких признаков отобрано, тем лучше модель
запоминает подсмотренное. Внутри конвейера отбор каждый раз делается
заново только на обучающей части, и честный результат держится около 0.5,
как и должно быть. Поэтому все шаги держат в одном `Pipeline`,
и кросс-валидация прогоняет его целиком.

### Подбор гиперпараметров и вложенная кросс-валидация

`GridSearchCV` — это **поиск**: он перебирает значения `C` и для каждого
считает кросс-валидацию. Для каждого `C` модель пять раз обучается на четырех
фолдах и оценивается на пятом, валидационном, и по среднему выбирается
лучшее `C`. Валидационные части здесь служат выбору, поэтому лучший
результат поиска — максимум из нескольких шумных чисел, и он оптимистичен.

Честную оценку дает **вложенная кросс-валидация** — кросс-валидация вокруг
всего поиска. Каждое ее разбиение откладывает свой тестовый сет. На остальных
данных поиск целиком, со своими валидационными частями, выбирает `C`
и обучает модель, а оценивается эта модель на тестовом сете разбиения,
которого поиск не видел ни разу. Результат — среднее по пяти таким тестовым
сетам, честная оценка всей процедуры «подобрать и обучить».

Схема на тех же игрушечных 20 объектах для первого разбиения вложенной
кросс-валидации. Верхняя строка — само разбиение: тестовый сет и все
остальное. Под чертой — пять разбиений поиска. Тестовый сет в поиске
не участвует, поэтому ниже черты его клетки пустые.

In [ ]:
outer_train_idx, outer_test_idx = next(KFold(5).split(toy_X))
inner_splits = list(KFold(5).split(outer_train_idx))

fig, ax = plt.subplots(figsize=(8.5, 3.4))
for i in outer_train_idx:
    draw_square(ax, i, 0, TRAIN_COLOR, edge=TRAIN_EDGE)
for i in outer_test_idx:
    draw_square(ax, i, 0, OCHRE)
labels = ["вложенная кросс-валидация,\nразбиение 1"]
for row, (fit_idx, check_idx) in enumerate(inner_splits, start=1):
    for i in outer_train_idx[fit_idx]:
        draw_square(ax, i, row, TRAIN_COLOR, edge=TRAIN_EDGE)
    for i in outer_train_idx[check_idx]:
        draw_square(ax, i, row, BLUE)
    labels.append(f"поиск C, разбиение {row}")
ax.axhline(1, color=BLACK, lw=1.2)
style_scheme_axes(ax, labels, "Вложенная кросс-валидация, первое разбиение")
handles = [
    Patch(facecolor=TRAIN_COLOR, edgecolor=TRAIN_EDGE, lw=1.5, label="обучение"),
    Patch(color=BLUE, label="валидационная часть поиска: выбор C"),
    Patch(color=OCHRE, label="тестовый сет разбиения: оценка"),
]
fig.legend(handles=handles, loc="upper center", ncol=3, frameon=False, fontsize=9, bbox_to_anchor=(0.5, 1.04))
plt.tight_layout()
plt.show()

Поиск видит только объекты с 4 по 19. На них он пять раз меняет
валидационную часть, выбирает `C` по среднему, обучает модель с этим `C`
на всех объектах с 4 по 19 и только потом один раз оценивает ее на тестовом
сете, объектах 0–3. Потом то же повторяется для остальных четырех разбиений
вложенной кросс-валидации, каждый раз со своим поиском, и выбранное `C`
от разбиения к разбиению может отличаться. В коде это одна строка
`cross_val_score(search, ...)`: вместо модели в нее передан сам `GridSearchCV`.

Для сравнения сначала обычный поиск: `GridSearchCV` на всех клиентах,
без вложенной кросс-валидации вокруг. Его валидационные части устроены
так же, как на схеме под чертой, только режут все данные. Дальше называем
его поиском на всех данных. Для каждого `C` — среднее и разброс ROC-AUC
по пяти валидационным частям.

In [ ]:
search_model = make_model(LogisticRegression(max_iter=2000))
search = GridSearchCV(
    search_model, param_grid={"logisticregression__C": [0.001, 0.01, 0.1, 1, 10]},
    cv=cv_churn, scoring="roc_auc",
)
search.fit(feature_frame, target)
grid_table = pd.DataFrame({
    "C": [f"{c:g}" for c in search.cv_results_["param_logisticregression__C"]],
    "ROC-AUC, среднее": search.cv_results_["mean_test_score"],
    "ROC-AUC, разброс": search.cv_results_["std_test_score"],
}).set_index("C")
show_table(grid_table)

Теперь вложенная кросс-валидация. Она делит всех клиентов на пять фолдов.
В каждом разбиении один фолд — тестовый сет, а на остальных четырех поиск
запускается заново, со своими валидационными частями: `cross_val_score`
на каждом разбиении делает свежую копию `search` с теми же настройками
и обучает ее с нуля. То, что `search` уже обучен на всех данных, здесь
ни на что не влияет. Другой `random_state` здесь не обязателен: поиск режет
только четыре пятых данных, и его валидационные части все равно
не совпадут с тестовыми сетами вложенной кросс-валидации.

In [ ]:
outer_cv = StratifiedKFold(5, shuffle=True, random_state=SEED + 1)
outer = cross_val_score(search, feature_frame, target, cv=outer_cv, scoring="roc_auc")
print("ROC-AUC на пяти тестовых сетах вложенной кросс-валидации:", outer.round(3))
print(f"лучший C: {search.best_params_['logisticregression__C']}, "
      f"лучший результат поиска на всех данных {search.best_score_:.3f}, "
      f"вложенная кросс-валидация {outer.mean():.3f} ± {outer.std():.3f}")

На графике синие точки — поиск на всех данных, по точке на каждое `C`.
Звездочка — та из них, которую выбрал поиск. Горизонтальная линия
с полосой разброса — вложенная кросс-валидация.

In [ ]:
plt.figure(figsize=(8, 3.8))
plt.errorbar(range(len(grid_table)), grid_table["ROC-AUC, среднее"], yerr=grid_table["ROC-AUC, разброс"],
             fmt="o-", color=BLUE, capsize=4, label="поиск на всех данных: среднее и разброс для каждого C")
plt.scatter(search.best_index_, search.best_score_, s=220, marker="*", color=BLACK, zorder=3,
            label=f"выбранное C, лучший результат поиска: {search.best_score_:.3f}")
plt.axhline(outer.mean(), color=OCHRE, lw=2, label=f"вложенная кросс-валидация, честная оценка: {outer.mean():.3f}")
plt.axhspan(outer.mean() - outer.std(), outer.mean() + outer.std(), color=OCHRE, alpha=0.15)
plt.xticks(range(len(grid_table)), grid_table.index)
plt.xlabel("C")
plt.ylabel("ROC-AUC")
plt.legend(fontsize=9)
plt.show()

Здесь разницы нет: при `C` от 0.1 до 10 модели почти одинаковые, их оценки
посчитаны на одних и тех же частях и отличаются меньше чем на тысячную.
Какую из них ни выбери, результат почти тот же, поэтому максимум почти
не завышен. Оптимизм становится виден, когда вариантов много, а данных мало. Вернемся к шумовым данным,
где честный ответ известен — 0.5, и будем подбирать сразу число отобранных
признаков и `C`, увеличивая сетку.

In [ ]:
noise_grids = [
    {"selectkbest__k": [20], "logisticregression__C": [1]},
    {"selectkbest__k": [5, 20, 50], "logisticregression__C": [0.1, 1]},
    {"selectkbest__k": [5, 10, 20, 50], "logisticregression__C": [0.01, 0.1, 1]},
    {"selectkbest__k": [5, 10, 20, 50, 100, 200], "logisticregression__C": [0.01, 0.1, 1, 10, 100]},
]
grid_sizes, inner_best, nested_mean = [], [], []
for grid in noise_grids:
    noise_pipeline = make_pipeline(SelectKBest(f_classif), LogisticRegression(max_iter=1000))
    noise_search = GridSearchCV(noise_pipeline, grid, cv=5, scoring="accuracy")
    noise_search.fit(X_noise, y_noise)
    noise_nested = cross_val_score(noise_search, X_noise, y_noise, cv=5, scoring="accuracy")
    grid_sizes.append(len(grid["selectkbest__k"]) * len(grid["logisticregression__C"]))
    inner_best.append(noise_search.best_score_)
    nested_mean.append(noise_nested.mean())

plt.figure(figsize=(8, 3.8))
plt.plot(grid_sizes, inner_best, "o-", color=OCHRE, lw=2, label="лучший результат поиска на всех данных")
plt.plot(grid_sizes, nested_mean, "o-", color=BLUE, lw=2, label="вложенная кросс-валидация")
plt.axhline(0.5, color=BLACK, ls="--", lw=1, label="честный ответ: случайное угадывание")
plt.xscale("log")
plt.xticks(grid_sizes, grid_sizes)
plt.xlabel("сколько вариантов гиперпараметров перебрано")
plt.ylabel("accuracy")
plt.title("Шумовые данные: чем больше перебор, тем больше самообман")
plt.legend(fontsize=9)
plt.show()

Лучший результат поиска на всех данных растет вместе с сеткой:
из большего числа шумных оценок выбирается все более удачная. Вложенная
кросс-валидация держится около 0.5 — это честная оценка всей процедуры
«подобрать и обучить».

## 9. Сохранение модели

Обученная модель — это объект Python. Чтобы использовать ее завтра, в другом
скрипте или на другом компьютере, ее сохраняют на диск. Для sklearn
стандарт — `joblib`. Сохранять надо весь конвейер: если сохранить только
логистическую регрессию, при загрузке придется заново повторить
масштабирование и кодирование, ровно с теми же параметрами.

In [ ]:
final_model = make_model(LogisticRegression(max_iter=2000))
final_model.fit(X_train, y_train)

In [ ]:
os.makedirs("artifacts", exist_ok=True)
model_path = os.path.join("artifacts", "churn_model.joblib")
joblib.dump(final_model, model_path)
print(f"сохранено: {model_path}, {os.path.getsize(model_path) / 1024:.0f} КБ")

In [ ]:
loaded = joblib.load(model_path)
same = np.allclose(loaded.predict_proba(X_test), final_model.predict_proba(X_test))
print("предсказания после загрузки совпадают:", same)

In [ ]:
new_client = X_test.iloc[[0]]
display(new_client)
print(f"вероятность ухода: {loaded.predict_proba(new_client)[0, 1]:.3f}")

Загруженный конвейер принимает сырую таблицу с теми же столбцами и сам
делает всю предобработку. Кроме самой модели полезно сохранить рядом
версии библиотек: `joblib`-файл, сохраненный в одной версии sklearn,
не обязан открываться в другой. И метрики, с которыми модель уходила
в работу, — чтобы было с чем сравнивать.

## 10. Тексты

Второй датасет: сообщения из новостных групп Usenet на пять тем.
Заголовки, подписи и цитаты убираем, чтобы модель училась на тексте,
а не на адресах авторов.

In [ ]:
topics = ["comp.graphics", "rec.autos", "rec.sport.hockey", "sci.space", "talk.politics.misc"]
news = fetch_20newsgroups(subset="all", categories=topics, remove=("headers", "footers", "quotes"), random_state=SEED)
texts = pd.DataFrame({"text": news.data, "topic": [news.target_names[t] for t in news.target]})
texts["topic"].value_counts()

EDA текстов начинается с того же, что и у таблиц: классы, пропуски,
дубли, длины.

In [ ]:
texts["length"] = texts["text"].str.len()
texts["words"] = texts["text"].str.split().str.len()
print(f"пустых или почти пустых (меньше 20 символов): {(texts['length'] < 20).sum()}")
print(f"точных дубликатов текста: {texts['text'].duplicated().sum()}")
print(f"из них непустых, от 20 символов: {(texts['text'].duplicated() & (texts['length'] >= 20)).sum()}")

In [ ]:
texts.sample(3, random_state=SEED)

Пустые тексты — это сообщения, где была одна цитата: после удаления цитат
ничего не осталось. Почти все «дубликаты» — это та же пустая строка,
настоящих повторов непустых сообщений всего три. Их все равно убираем:
копия одного сообщения в обучении и в тестовом сете завышает качество.

In [ ]:
texts = texts[texts["length"] >= 20].drop_duplicates("text").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9, 3.8))
for topic in topics:
    ax.hist(np.log10(texts.loc[texts["topic"] == topic, "words"]), bins=40, histtype="step", lw=1.8, label=topic)
ax.set_xlabel("log10 числа слов")
ax.set_title("Длины сообщений: от одной строки до статьи")
ax.legend(fontsize=9)
plt.show()
print("осталось текстов:", len(texts))

### Токенизация, стоп-слова, стемминг

Первый шаг — разбить текст на слова, токены. Разделить по пробелам мало:
знаки препинания прилипнут к словам. В `nltk` для этого есть
`wordpunct_tokenize`: он по регулярному выражению отделяет слова от знаков
препинания и одинаково работает для любого языка. Есть и более умный
`word_tokenize`, но ему нужен отдельный скачиваемый словарь.
Дальше обычно выбрасывают стоп-слова — предлоги, союзы,
местоимения, которые есть в любом тексте, — и приводят слова к общей
основе, чтобы «газета», «газеты» и «газетами» стали одним признаком.
Стемминг просто отрезает окончания по правилам, лемматизация приводит
к словарной форме, но ей нужен морфологический словарь.

Списки стоп-слов `nltk` скачиваются один раз. Функция ниже скачивает словарь,
только если его еще нет на диске, чтобы не ходить в интернет при каждом
запуске. Если скачать не получилось, она пишет одну понятную строку вместо
длинной ошибки. Частая причина — прокси: `nltk` из соображений безопасности
не качает через него.

In [ ]:
def ensure_nltk(resource, path):
    try:
        nltk.data.find(path)
    except LookupError:
        downloaded = nltk.download(resource, quiet=True, print_error_to=io.StringIO())
        if not downloaded:
            raise RuntimeError(
                f"не удалось скачать словарь nltk «{resource}». Если в системе настроен прокси, "
                "запустите эту ячейку один раз без него: nltk не качает через прокси"
            )


ensure_nltk("stopwords", "corpora/stopwords")

Разберем одно русское предложение по шагам. Сначала токены.

In [ ]:
sentence_ru = "Мы читали газеты, а в газетах писали о новых космических кораблях."
tokens_ru = wordpunct_tokenize(sentence_ru.lower())
tokens_ru

Выбрасываем знаки препинания и стоп-слова.

In [ ]:
stop_ru = set(stopwords.words("russian"))
content_ru = [token for token in tokens_ru if token.isalpha() and token not in stop_ru]
content_ru

Приводим к основам: «газеты» и «газетах» становятся одним и тем же.

In [ ]:
stemmer_ru = SnowballStemmer("russian")
[stemmer_ru.stem(token) for token in content_ru]

Для английского те же три шага, сразу в одной ячейке.

In [ ]:
sentence_en = "We were reading newspapers, and the papers wrote about new spacecraft launches."
tokens_en = wordpunct_tokenize(sentence_en.lower())
stop_en = set(stopwords.words("english"))
stemmer_en = SnowballStemmer("english")
content_en = [token for token in tokens_en if token.isalpha() and token not in stop_en]
print("токены:       ", tokens_en)
print("без стоп-слов:", content_en)
print("основы:       ", [stemmer_en.stem(token) for token in content_en])

### Мешок слов

Самое простое представление текста: словарь всех слов корпуса, и для
каждого текста — сколько раз в нем встретилось каждое слово. Порядок слов
теряется, отсюда название. По сути это one-hot из шестого раздела,
только для каждого слова текста, и сложенный по всем словам.

In [ ]:
corpus = [
    "the rocket launch was delayed",
    "the hockey game was delayed by rain",
    "a new rocket engine for the launch",
]
bag = CountVectorizer()
toy_matrix = bag.fit_transform(corpus)
toy_counts = pd.DataFrame(toy_matrix.toarray(), columns=bag.get_feature_names_out(), index=[f"текст {i}" for i in range(3)])
show_table(toy_counts, digits=0)

На настоящем корпусе словарь огромный, а в каждом тексте из него
встречается малая доля.

In [ ]:
word_counter = CountVectorizer()
news_counts = word_counter.fit_transform(texts["text"])
density = news_counts.nnz / (news_counts.shape[0] * news_counts.shape[1])
print(f"матрица {news_counts.shape[0]} × {news_counts.shape[1]}, ненулевых {density:.2%}")
print(f"в памяти как разреженная: {(news_counts.data.nbytes + news_counts.indices.nbytes + news_counts.indptr.nbytes) / 1e6:.1f} МБ, "
      f"как плотная была бы {news_counts.shape[0] * news_counts.shape[1] * 8 / 1e9:.1f} ГБ")

Такие матрицы хранят разреженными: только ненулевые значения и их
координаты. sklearn умеет обучать линейные модели прямо на них. Почему
словарь такой большой, видно по частотам слов: несколько слов встречаются
десятки тысяч раз, а большинство — один-два раза. Отсортируем слова
по частоте и нарисуем частоту от места в этом списке в логарифмических
осях — получится почти прямая, это закон Ципфа.

In [ ]:
word_totals = np.sort(np.asarray(news_counts.sum(axis=0)).ravel())[::-1]
ranks = np.arange(1, len(word_totals) + 1)

plt.figure(figsize=(8, 3.8))
plt.loglog(ranks, word_totals, color=BLUE)
plt.xlabel("место слова в списке по частоте")
plt.ylabel("сколько раз встретилось")
plt.title("Частоты слов в корпусе")
plt.show()
print(f"слов, встретившихся один раз: {(word_totals == 1).mean():.0%} словаря")

### N-граммы

Мешок слов видит слово «not», но не знает, к чему оно относится:
у отзывов «good, not bad» и «bad, not good» мешки слов одинаковые. **N-грамма** — это $n$ слов подряд. Униграммы — отдельные слова,
биграммы — пары соседних слов, триграммы — тройки. Разберем одно
предложение.

In [ ]:
phrase_tokens = "the movie is not good".split()
phrase_tokens

Биграммы — каждое слово в паре со следующим. `zip(tokens, tokens[1:])`
ставит рядом каждое слово и следующее за ним.

In [ ]:
[" ".join(pair) for pair in zip(phrase_tokens, phrase_tokens[1:])]

Триграммы — тройки подряд.

In [ ]:
[" ".join(triple) for triple in zip(phrase_tokens, phrase_tokens[1:], phrase_tokens[2:])]

Теперь те два отзыва. Смысл у них противоположный, а строки мешка слов
совпадают полностью.

In [ ]:
two_reviews = ["good, not bad", "bad, not good"]
unigram_bag = CountVectorizer(ngram_range=(1, 1))
unigram_counts = pd.DataFrame(unigram_bag.fit_transform(two_reviews).toarray(), columns=unigram_bag.get_feature_names_out(), index=two_reviews)
show_table(unigram_counts, digits=0)

С `ngram_range=(1, 2)` признаками становятся и слова, и биграммы.
Появляются столбцы «not bad» и «not good», и строки наконец различаются.

In [ ]:
bigram_bag = CountVectorizer(ngram_range=(1, 2))
bigram_counts_toy = pd.DataFrame(bigram_bag.fit_transform(two_reviews).toarray(), columns=bigram_bag.get_feature_names_out(), index=two_reviews)
show_table(bigram_counts_toy, digits=0)

Осторожно со стоп-словами. В списке стоп-слов sklearn есть «not» и «no»,
и вместе с ними пропадает отрицание. К тому же биграммы строятся уже
после удаления стоп-слов, из слов, которые в тексте стояли не рядом.

In [ ]:
stop_bag = CountVectorizer(ngram_range=(1, 2), stop_words="english")
stop_counts = pd.DataFrame(stop_bag.fit_transform(two_reviews).toarray(), columns=stop_bag.get_feature_names_out(), index=two_reviews)
show_table(stop_counts, digits=0)

«Not» исчез, а биграммы «good bad» и «bad good» склеены из слов, которые
соседями не были. Для задач, где отрицание важно, например для отзывов
или реакции пользователя, список стоп-слов либо не используют, либо
убирают из него отрицания.

Цена — размер словаря. Посмотрим, сколько признаков получается на новостях
при разной длине n-грамм. Можно добавить в список `(2, 2)` — только биграммы —
и сравнить.

In [ ]:
vocabulary_sizes = {}
for low, high in [(1, 1), (1, 2), (1, 3)]:
    ngram_counter = CountVectorizer(ngram_range=(low, high))
    ngram_counter.fit(texts["text"])
    vocabulary_sizes[f"n от {low} до {high}"] = len(ngram_counter.vocabulary_)

plt.figure(figsize=(7, 3.2))
bars = plt.bar(list(vocabulary_sizes), list(vocabulary_sizes.values()), color=BLUE)
plt.bar_label(bars, labels=[f"{v:,}".replace(",", " ") for v in vocabulary_sizes.values()])
plt.ylabel("признаков в словаре")
plt.show()

Самые частые биграммы после удаления стоп-слов уже похожи на темы корпуса.

In [ ]:
bigrams = CountVectorizer(ngram_range=(2, 2), stop_words="english", min_df=5)
bigram_counts = bigrams.fit_transform(texts["text"])
frequent = pd.Series(np.asarray(bigram_counts.sum(axis=0)).ravel(), index=bigrams.get_feature_names_out())
top_bigrams = frequent.sort_values().tail(15)

plt.figure(figsize=(8, 4.5))
plt.barh(top_bigrams.index, top_bigrams.to_numpy(), color=BLUE)
plt.xlabel("сколько раз встретилась")
plt.title("Самые частые биграммы после удаления стоп-слов")
plt.show()

Первая биграмма «don know» — это «don't know». Стандартная токенизация
sklearn берет только токены из двух и больше букв или цифр, поэтому
от «don't» остается «don», а «t» выпадает. Такие мелочи всплывают только тогда,
когда смотришь на данные глазами.

### TF-IDF: откуда он берется

Сначала обозначения. Текст $d$ — последовательность слов $w_1, w_2, \dots, w_{L_d}$,
где $L_d$ — длина текста в словах. Слово из словаря обозначим $t$. Сколько раз
слово $t$ встретилось в тексте $d$ — это

$$n_{t,d} = \sum_{k=1}^{L_d} [w_k = t],$$

где $[w_k = t]$ равно 1, если $k$-е слово текста — это $t$, и 0 иначе. Если
сложить эти счетчики по всем словам словаря, каждое слово текста посчитается
ровно один раз, и получится длина:

$$\sum_{t'} n_{t',d} = L_d.$$

Например, в тексте «the cat sat on the mat» $L_d = 6$, $n_{\text{the},d} = 2$,
$n_{\text{cat},d} = 1$, а слова, которых в тексте нет, дают $n_{t,d} = 0$.
В мешке слов признак $t$ у текста $d$ — это как раз $n_{t,d}$, сырая частота.
У такого признака две беды, разберем их по очереди на новостях.

**Беда первая: сырая частота растет с длиной текста.** Возьмем служебное
слово «the» и тематическое «space» и нарисуем, сколько раз каждое
встретилось, в зависимости от длины текста в словах.

In [ ]:
doc_length = np.asarray(news_counts.sum(axis=1)).ravel()
word_index = word_counter.vocabulary_
the_counts = news_counts[:, word_index["the"]].toarray().ravel()
space_counts = news_counts[:, word_index["space"]].toarray().ravel()

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for ax, (word, values) in zip(axes, [("the", the_counts), ("space", space_counts)]):
    ax.scatter(doc_length, values, s=5, alpha=0.3, color=BLUE)
    ax.set_xscale("log")
    ax.set_xlabel("длина текста в словах")
    ax.set_ylabel(f"сколько раз «{word}»")
    ax.set_title(f"Сырая частота «{word}»")
plt.tight_layout()
plt.show()

«The» в длинной статье встречается сотни раз, в короткой реплике — пару раз.
Сырая частота в основном измеряет длину текста, а не то, о чем он.
Поделим счетчик на длину текста:

$$\mathrm{tf}(t, d) = \frac{n_{t,d}}{L_d} = \frac{n_{t,d}}{\sum_{t'} n_{t',d}}.$$

Это **частота слова в тексте**, term frequency: какая доля слов текста
приходится на слово $t$. Сумма по всем словам словаря равна 1. В примере
«the cat sat on the mat» $\mathrm{tf}(\text{the}, d) = \frac{2}{6} \approx 0.33$.

In [ ]:
has_words = doc_length > 0
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for ax, (word, values) in zip(axes, [("the", the_counts), ("space", space_counts)]):
    ax.scatter(doc_length[has_words], values[has_words] / doc_length[has_words], s=5, alpha=0.3, color=OCHRE)
    ax.set_xscale("log")
    ax.set_xlabel("длина текста в словах")
    ax.set_ylabel(f"доля «{word}» в тексте")
    ax.set_title(f"Частота «{word}», деленная на длину")
plt.tight_layout()
plt.show()

Доля «the» теперь почти не зависит от длины: около пяти процентов в любом
тексте, у коротких текстов просто больше разброс.

**Беда вторая: чаще всего встречаются слова, которые ничего не говорят
о теме.** Посмотрим, у каких слов самая большая средняя доля по всем
текстам.

In [ ]:
length_share = news_counts[has_words].multiply(1 / doc_length[has_words][:, None]).tocsr()
mean_share = pd.Series(np.asarray(length_share.mean(axis=0)).ravel(), index=word_counter.get_feature_names_out())
top_share = mean_share.sort_values().tail(15)

plt.figure(figsize=(8, 4.5))
plt.barh(top_share.index, top_share.to_numpy(), color=OCHRE)
plt.xlabel("средняя доля слова в тексте")
plt.title("Самые частые слова: одни служебные")
plt.show()

Все пятнадцать — служебные слова. Их можно выкинуть списком стоп-слов,
но есть способ тоньше: понизить вес слова тем сильнее, чем в большем числе
текстов оно встречается. Пусть $n$ — число текстов в корпусе.
**Документная частота** $\mathrm{df}(t)$ — в скольких текстах слово $t$
встретилось хотя бы раз:

$$\mathrm{df}(t) = \sum_{d=1}^{n} [n_{t,d} > 0].$$

Сколько раз слово повторяется внутри одного текста, здесь неважно: текст
с сотней «the» и текст с одним «the» добавляют к $\mathrm{df}$ по единице. Слово,
которое есть в каждом тексте, не помогает отличать тексты друг от друга;
слово из одного текста — помогает сильно. Отсюда обратная документная
частота, в варианте sklearn:

$$\mathrm{idf}(t) = \ln\frac{1 + n}{1 + \mathrm{df}(t)} + 1$$

Единицы в дроби — сглаживание: как будто есть еще один текст, в котором
встречается каждое слово, поэтому деления на ноль не бывает. Логарифм
сжимает шкалу. Подставим слово из одного текста, $\mathrm{df}(t) = 1$:

$$\mathrm{idf}(t) = \ln\frac{1 + n}{1 + 1} + 1 = \ln\frac{1 + n}{2} + 1 \approx \ln n - \ln 2 + 1.$$

На наших 4553 текстах это около 8.7, а без логарифма было бы порядка $n / 2$,
больше двух тысяч. Единица после логарифма нужна, чтобы слово, которое есть
во всех текстах, получило вес 1, а не 0: при $\mathrm{df}(t) = n$ дробь равна 1,
логарифм — 0.

In [ ]:
n_news = news_counts.shape[0]
news_doc_freq = np.asarray((news_counts > 0).sum(axis=0)).ravel()
idf_values = np.log((1 + n_news) / (1 + news_doc_freq)) + 1
df_share = news_doc_freq / n_news

order = np.argsort(df_share)
plt.figure(figsize=(8, 3.8))
plt.plot(df_share[order], idf_values[order], color=BLUE, lw=2)
for word in ["the", "space", "hockey", "shuttle"]:
    k = word_index[word]
    plt.scatter(df_share[k], idf_values[k], color=OCHRE, zorder=3)
    plt.annotate(word, (df_share[k], idf_values[k]), textcoords="offset points", xytext=(5, 5))
plt.xscale("log")
plt.xlabel("доля текстов, где есть слово: df / n")
plt.ylabel("idf")
plt.title("Чем в большем числе текстов слово, тем меньше его вес")
plt.show()

Вес слова в тексте — частота, умноженная на idf:

$$w(t, d) = \mathrm{tf}(t, d) \cdot \mathrm{idf}(t)$$

В sklearn за $\mathrm{tf}$ берется сырая частота $n_{t,d}$, а зависимость от длины
убирает последний шаг: каждую строку делят на ее евклидову норму.

$$\hat{w}(t, d) = \frac{w(t, d)}{\sqrt{\sum_{t'} w(t', d)^2}}$$

Почему делить на длину текста тогда не нужно. Если взять
$\mathrm{tf}(t, d) = \frac{n_{t,d}}{L_d}$ вместо сырой частоты, каждый вес строки
умножится на одно и то же число $\frac{1}{L_d}$:

$$w'(t, d) = \frac{n_{t,d}}{L_d}\,\mathrm{idf}(t) = \frac{1}{L_d}\,w(t, d).$$

Норма строки умножится на то же число, $\sqrt{\sum_{t'} w'(t', d)^2} = \frac{1}{L_d}\sqrt{\sum_{t'} w(t', d)^2}$,
и в частном $\frac{1}{L_d}$ сократится: $\hat{w}$ получится тем же, что с сырой
частотой. Нормировка по евклидовой норме делает то же, что деление на длину.
Но вес слова от длины текста все-таки зависит: в тексте из $L$ разных слов,
каждое по одному разу и с одинаковым idf, все веса равны $\frac{1}{\sqrt{L}}$.
Чем больше в тексте разных слов, тем меньше вес каждого.
Соберем это в функцию по матрице частот.

In [ ]:
def tfidf(count_matrix: np.ndarray) -> np.ndarray:
    dense = np.asarray(count_matrix, dtype=float)
    n_texts = dense.shape[0]
    text_freq = (dense > 0).sum(axis=0)
    idf = np.log((1 + n_texts) / (1 + text_freq)) + 1
    weights = dense * idf
    norms = np.linalg.norm(weights, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    normalized = weights / norms
    return normalized

На маленьком корпусе из трех текстов сравним с `TfidfVectorizer`.

In [ ]:
mine = tfidf(toy_matrix.toarray())
reference = TfidfVectorizer().fit_transform(corpus).toarray()
print("наибольшее расхождение с TfidfVectorizer:", np.abs(mine - reference).max())
show_table(pd.DataFrame(mine, columns=bag.get_feature_names_out(), index=[f"текст {i}" for i in range(3)]), digits=2)

Слово «the», которое есть во всех трех текстах, в каждом тексте весит
меньше, чем слова, встретившиеся в одном тексте: «hockey», «engine», «rain».

Что TF-IDF делает на новостях. Посчитаем его по всему корпусу и сначала
проверим, в скольких текстах самым весомым оказывается «the».

In [ ]:
news_tfidf = TfidfVectorizer().fit_transform(texts["text"])
the_column = word_index["the"]
top_by_count = np.asarray(news_counts.argmax(axis=1)).ravel()
top_by_tfidf = np.asarray(news_tfidf.argmax(axis=1)).ravel()
print(f"«the» — самое частое слово в {np.mean(top_by_count[has_words] == the_column):.0%} текстов")
print(f"«the» — самое весомое по TF-IDF в {np.mean(top_by_tfidf[has_words] == the_column):.0%} текстов")

В сырых частотах «the» на первом месте почти в половине текстов, в TF-IDF —
в нескольких процентах. Слово, встреченное один раз, с idf около 8
перевешивает пять «the» с idf около 1.1.

Теперь зависимость от длины. Разложим тексты, где есть «the», на группы
по длине и в каждой группе возьмем медиану веса «the» в трех представлениях:
доля слова в тексте $\mathrm{tf}(t, d)$, обычный TF-IDF и TF-IDF
с `sublinear_tf=True`, где сырая частота $n_{t,d}$ заменена на $1 + \ln n_{t,d}$.

In [ ]:
news_tfidf_sublinear = TfidfVectorizer(sublinear_tf=True).fit_transform(texts["text"])
length_edges = [1, 30, 100, 300, 1000, 20000]
length_labels = ["до 30", "30–100", "100–300", "300–1000", "больше 1000"]
the_present = the_counts > 0
representations = {
    "доля в тексте": the_counts / np.maximum(doc_length, 1),
    "TF-IDF": news_tfidf[:, the_column].toarray().ravel(),
    "TF-IDF, sublinear_tf": news_tfidf_sublinear[:, the_column].toarray().ravel(),
}

plt.figure(figsize=(9, 3.8))
for (name, values), color in zip(representations.items(), [GREY, BLUE, OCHRE]):
    medians = [np.median(values[the_present & (doc_length >= low) & (doc_length < high)])
               for low, high in zip(length_edges[:-1], length_edges[1:])]
    plt.plot(length_labels, medians, "o-", color=color, lw=2, label=name)
plt.xlabel("длина текста в словах")
plt.ylabel("медиана веса «the»")
plt.title("Вес «the» в текстах разной длины")
plt.legend()
plt.show()

Доля «the» в тексте почти не зависит от длины. Вес TF-IDF — зависит,
и растет: L2-нормировка уравнивает длину вектора, а не вклад отдельных
слов. В длинном тексте много слов, встреченных по одному разу, у каждого
маленький вес, и частое слово забирает большую долю нормы. С логарифмом
частоты все наоборот: сто повторений весят не в сто раз больше одного,
а примерно в 5.6 раза, и в длинных текстах «the» весит меньше. Какой
вариант лучше для задачи, решает эксперимент; `sublinear_tf` — еще одно
представление, которое стоит попробовать.

Теперь посчитаем, какую долю суммарного веса текста несут стоп-слова
из списка `nltk`. Пусть $S$ — множество стоп-слов, $w(t, d)$ — вес слова
в тексте: сырая частота $n_{t,d}$ или TF-IDF. Доля стоп-слов в тексте $d$

$$\mathrm{share}(d) = \frac{\sum_{t \in S} w(t, d)}{\sum_{t} w(t, d)},$$

на графике — ее среднее по всем текстам.

In [ ]:
vocabulary_all = word_counter.get_feature_names_out()
is_stop = np.array([word in stop_en for word in vocabulary_all])


def stop_share(matrix):
    total = np.asarray(matrix.sum(axis=1)).ravel()
    on_stop = np.asarray(matrix[:, is_stop].sum(axis=1)).ravel()
    share = float(np.mean(on_stop[total > 0] / total[total > 0]))
    return share


shares = [stop_share(news_counts), stop_share(news_tfidf)]
plt.figure(figsize=(5, 3.6))
bars = plt.bar(["сырые частоты", "TF-IDF"], shares, color=[GREY, BLUE])
plt.bar_label(bars, labels=[f"{v:.0%}" for v in shares])
plt.ylim(0, 0.6)
plt.title("Доля веса текста на стоп-словах")
plt.show()

В сырых частотах стоп-слова несут 43% веса текста, в TF-IDF — 22%, почти
вдвое меньше.

Теперь отдельные слова в текстах темы `sci.space`: десять служебных
и пять тематических. Слева — средний вес слова, если вес «the» принять
за 1, в сырых частотах и в TF-IDF. Справа — во сколько раз TF-IDF поднял
это отношение, в зависимости от idf слова. Пунктир — единица, то есть
«не изменилось».

In [ ]:
space_rows = (texts["topic"] == "sci.space").to_numpy()
mean_count = np.asarray(news_counts[space_rows].mean(axis=0)).ravel()
mean_tfidf = np.asarray(news_tfidf[space_rows].mean(axis=0)).ravel()
compare_words = ["of", "to", "and", "in", "is", "that", "it", "for", "on", "this",
                 "space", "nasa", "orbit", "launch", "shuttle"]
compare_index = [word_index[w] for w in compare_words]
relative_count = mean_count[compare_index] / mean_count[word_index["the"]]
relative_tfidf = mean_tfidf[compare_index] / mean_tfidf[word_index["the"]]
growth = relative_tfidf / relative_count
compare_idf = idf_values[compare_index]
is_stop_word = np.array([w in stop_en for w in compare_words])

fig, axes = plt.subplots(1, 2, figsize=(15, 4.2), gridspec_kw={"width_ratios": [3, 2]})
positions = np.arange(len(compare_words))
axes[0].bar(positions - 0.2, relative_count, width=0.4, color=GREY, label="сырые частоты")
axes[0].bar(positions + 0.2, relative_tfidf, width=0.4, color=BLUE, label="TF-IDF")
axes[0].set_xticks(positions, compare_words, rotation=45)
axes[0].set_title("sci.space: средний вес слова, если вес «the» принять за 1")
axes[0].legend()
for mask, color, label in [(is_stop_word, OCHRE, "стоп-слова"), (~is_stop_word, "#4DAF4A", "тематические")]:
    axes[1].scatter(compare_idf[mask], growth[mask], color=color, s=40, label=label, zorder=3)
for word, x, y in zip(compare_words, compare_idf, growth):
    axes[1].annotate(word, (x, y), xytext=(4, 3), textcoords="offset points", fontsize=8)
axes[1].axhline(1, color=BLACK, ls="--", lw=1)
axes[1].set_xlabel("idf слова")
axes[1].set_ylabel("во сколько раз TF-IDF поднял вес")
axes[1].set_title("Прибавка растет вместе с idf")
axes[1].legend()
plt.tight_layout()
plt.show()

Чем реже слово встречается в корпусе, тем больше его idf и тем сильнее
TF-IDF поднимает его вес относительно «the»:

- «of», «to», «and» есть в 70–76% текстов, idf у них от 1.3 до 1.4,
  и прибавляют они в 1.05–1.2 раза;
- «in», «is», «that», «it», «for», «on», «this» есть в 45–66% текстов,
  idf от 1.4 до 1.8, прибавка в 1.2–2 раза;
- тематические «space», «nasa», «orbit», «launch», «shuttle» есть в 3–9%
  текстов, idf от 3.5 до 4.6, прибавка в 2.5–4 раза.

Четкой границы между служебными и тематическими словами TF-IDF не проводит:
вес плавно растет с редкостью слова: служебное «it» прибавляет в 1.96 раза,
тематическое «space» — в 2.53. Поэтому служебные слова убирают отдельно,
списком стоп-слов. А какие слова отличают тему, окончательно выбирает
модель: ниже видно, что логистическая регрессия на TF-IDF для темы космоса
ставит на первые места «space», «shuttle», «launch», «nasa».

### Классификация тем

Сравниваем представления в одном сетапе: одно разбиение, одна модель,
одни метрики. Классов пять, поэтому F1 усредняем по классам, как описано
в разделе 7: micro-F1 совпадает с accuracy, macro-F1 дает каждой теме
одинаковый вес.

In [ ]:
text_train, text_test, topic_train, topic_test = train_test_split(
    texts["text"], texts["topic"], test_size=0.25, random_state=SEED, stratify=texts["topic"]
)
text_results = {}


def evaluate_text(name, vectorizer):
    model = make_pipeline(vectorizer, LogisticRegression(max_iter=3000))
    model.fit(text_train, topic_train)
    predicted = model.predict(text_test)
    text_results[name] = {
        "признаков": len(model[0].vocabulary_),
        "accuracy": accuracy_score(topic_test, predicted),
        "micro-F1": f1_score(topic_test, predicted, average="micro"),
        "macro-F1": f1_score(topic_test, predicted, average="macro"),
    }
    return model


evaluate_text("мешок слов", CountVectorizer(min_df=2))
evaluate_text("мешок слов без стоп-слов", CountVectorizer(min_df=2, stop_words="english"))
evaluate_text("TF-IDF", TfidfVectorizer(min_df=2, stop_words="english"))
text_model = evaluate_text("TF-IDF, слова и биграммы", TfidfVectorizer(min_df=2, stop_words="english", ngram_range=(1, 2)))
text_table = pd.DataFrame(text_results).T
text_table["признаков"] = text_table["признаков"].astype(int)
show_table(text_table)

In [ ]:
plot_metric_bars(text_table[["accuracy", "micro-F1", "macro-F1"]], "Четыре представления текста на одном тестовом сете")

TF-IDF заметно лучше сырых частот: логистическая регрессия получает
признаки одного масштаба, и частые пустые слова ее больше не отвлекают.
Micro-F1 совпадает с accuracy, как и обещано.

### Какие слова модель считает главными

У логистической регрессии для каждой темы свой вектор весов, по весу
на каждое слово словаря. Достанем из конвейера оба шага.

In [ ]:
vectorizer = text_model[0]
classifier = text_model[1]
vocabulary = vectorizer.get_feature_names_out()
print("слов и биграмм в словаре:", len(vocabulary))
vocabulary[:10]

Веса лежат в матрице: строка — тема, столбец — слово словаря.

In [ ]:
class_weights = classifier.coef_
print("форма матрицы весов:", class_weights.shape)
print("темы по строкам:", list(classifier.classes_))

Для одной темы берем ее строку и сортируем слова по весу. `np.argsort`
возвращает номера столбцов от меньшего веса к большему, `[::-1]`
разворачивает порядок.

In [ ]:
space_row = list(classifier.classes_).index("sci.space")
space_row

In [ ]:
order = np.argsort(class_weights[space_row])[::-1]
order[:12]

Это номера столбцов, то есть номера слов в словаре. Сами слова и их веса:

In [ ]:
vocabulary[order[:12]]

In [ ]:
class_weights[space_row, order[:12]].round(2)

In [ ]:
top_space = pd.Series(class_weights[space_row, order[:12]], index=vocabulary[order[:12]])
plt.figure(figsize=(7, 4))
plt.barh(top_space.index[::-1], top_space.to_numpy()[::-1], color=BLUE)
plt.xlabel("вес в логистической регрессии")
plt.title("sci.space: слова с наибольшим весом")
plt.show()

То же самое для всех тем сразу: по десять слов с наибольшим весом.

In [ ]:
top_words = {}
for row, topic in enumerate(classifier.classes_):
    best = np.argsort(class_weights[row])[::-1][:10]
    top_words[topic] = vocabulary[best]
pd.DataFrame(top_words)

In [ ]:
predicted_topics = text_model.predict(text_test)
matrix = confusion_matrix(topic_test, predicted_topics, labels=classifier.classes_)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.imshow(matrix, cmap="Blues")
ax.set_xticks(range(len(topics)), classifier.classes_, rotation=30, ha="right")
ax.set_yticks(range(len(topics)), classifier.classes_)
for i in range(len(topics)):
    for j in range(len(topics)):
        ax.text(j, i, matrix[i, j], ha="center", va="center", color="white" if matrix[i, j] > matrix.max() / 2 else BLACK)
ax.set_xlabel("предсказано")
ax.set_ylabel("на самом деле")
ax.set_title("Матрица ошибок на тестовом сете")
ax.grid(False)
plt.tight_layout()
plt.show()

In [ ]:
errors = pd.DataFrame({"text": text_test, "true": topic_test, "predicted": predicted_topics})
errors = errors[errors["true"] != errors["predicted"]]
for _, row in errors.head(3).iterrows():
    print(f"на самом деле {row['true']}, предсказано {row['predicted']}:")
    print("   ", row["text"][:250].replace("\n", " "), "\n")

Ошибки стоит читать глазами. Часть из них — короткие сообщения, по которым
тему не угадал бы и человек; часть — тексты на стыке тем, например, про
графику для космических снимков. Разбор ошибок — обязательная часть любой
работы с моделью, и в домашней работе он тоже нужен.

## 11. Автоматический отчет по таблице

Большую часть того, что мы делали руками в разделах 2–5, умеют строить
библиотеки автоматического анализа. Самая известная — `fg-data-profiling`,
раньше она называлась `ydata-profiling`, а еще раньше `pandas-profiling`. По таблице она за один вызов
строит HTML-отчет: тип и сводку каждого столбца, пропуски, распределения,
самые частые значения, корреляции и взаимодействия признаков, дубликаты
и список предупреждений вроде «сильный дисбаланс» или «много уникальных
значений».

Построим отчет по сырой таблице `raw`: с нее мы начинали, и интересно
сравнить, что библиотека заметит сама. Отчет сохраняется в файл, откройте
его в браузере. Строится он несколько секунд.

In [ ]:
import contextlib

from data_profiling import ProfileReport

profile = ProfileReport(raw, title="Отток клиентов: сырые данные", progress_bar=False)
profile_path = os.path.join("artifacts", "telco_profile.html")
with contextlib.redirect_stderr(io.StringIO()):
    profile.to_file(profile_path)
print("отчет:", profile_path)

Отчет полезен как первый взгляд: за несколько секунд видно почти все, на что мы
потратили час. Но смотреть его надо с тем же вопросом «почему так».
Библиотека покажет, что у `TotalCharges` шесть с половиной тысяч разных
строковых значений, но не скажет, что одиннадцать из них — пробелы
у новых клиентов и что заполнять их надо нулем. Она покажет корреляции,
но не отличит связь от причины. Автоматический отчет подсказывает, куда
смотреть, а выводы по-прежнему делает человек.

## 12. Что запомнить

**Сначала смотреть, потом обучать.** Типы столбцов, `describe`, свои
сводки, пропуски, невозможные значения, дубли, случайная выборка строк.
Пустая строка вместо числа и кавычки внутри категорий ломают данные тихо,
без ошибок.

**Пропуск — тоже информация.** Механизм пропусков определяет, чем их
заполнять. Выброс и дубликат — гипотезы, которые надо проверить.

**Связь не причина.** Доля ушедших по одному признаку смешивает эффекты
связанных признаков; таблицы сопряженности и сводные таблицы помогают
их разделить.

**Категориальные признаки.** One-hot со свободным членом дает вырожденную
матрицу, поэтому один столбец выбрасывают или добавляют регуляризацию.
`OneHotEncoder` в конвейере вместо `get_dummies` по отдельности.
Target encoding — только с кросс-фиттингом: строки обучения кодируются
через `fit_transform`, тестовый сет — через `transform` долями по всему обучению.
Честная кодировка ведет себя на обучении так же, как на тестовом сете.

**Метрики.** Accuracy при дисбалансе обманывает. Precision, recall и F1
считаются по клеткам матрицы ошибок, F1 высокий только при высоких обоих.
ROC-AUC — вероятность правильного порядка пары, PR-AUC — средняя precision
по положительным объектам, честнее при редком классе. Для многих классов
macro-F1 видит редкие классы, micro-F1 равен accuracy.

**Кросс-валидация.** Одно разбиение — шум. Схему выбирают под данные:
стратификация, группы, время. Все, что учится на данных, живет внутри
`Pipeline`, иначе утечка.

**Модель сохраняют целиком**, вместе с предобработкой, через `joblib`.

**Тексты.** Токенизация, стоп-слова, стемминг, мешок слов, n-граммы.
TF-IDF в sklearn приводит каждый текст к вектору единичной длины и понижает
вес слов, которые есть везде. Стоп-слова sklearn удаляют и отрицания. Разреженные матрицы и линейная модель на них — сильный
baseline для классификации текстов.